# Part 2 — MYH Curated Applications Dataset (2020–2025)

**Notebook role:** This notebook is the full, rerunnable raw-to-curated workflow for Part 2 of the Data Pipeline Project.

**Current implementation status:**  
Sub-project **2.7** has now added the final SQL/API handoff note, the closing Part 2 reflection, and the definition-of-done checkpoint while preserving the validated 2.6 curated dataset semantics. The notebook still builds the same **7,641-row × 32-column** `curated_applications` table and keeps the paired CSV and Parquet export contract unchanged.  
The remaining final closure step is the full end-to-end rerun/export confirmation plus the synchronized repository-facing and handoff/control documentation update.

## Project purpose

The finished notebook:
- reads the original MYH Excel workbooks for application rounds **2020–2025**,
- builds a longitudinal curated applications dataset from **`Tabell 3`**,
- preserves a clear main-table grain: **one row = one application in one application round**,
- documents source differences, harmonization choices, cleaning choices, enrichment, validation, final reflection, and downstream handoff,
- exports a finished dataset that can later be loaded into SQL and served through a read-oriented API.


## 1. Scope and design principles

### Fixed project direction
- Source years: **2020, 2021, 2022, 2023, 2024, 2025**
- Main source sheet: **`Tabell 3`**
- Main table grain: **one application in one application round per row**
- `Tabell 4` is acknowledged as useful but must **not** be merged into the main applications table in a way that duplicates applications.

### How to read and rerun this notebook
Run the notebook from top to bottom. It discovers the raw Excel workbooks under `data/raw/`, profiles the relevant source structures, builds and validates the curated applications table, and then refreshes the finished processed exports under `data/processed/`.

No manual mid-run edits are intended. Repeated runs should follow the same raw-to-curated path, and processed files are refreshed only after the validation gate has passed.

### Working quality standard
This notebook should read as an explanatory data journey rather than a code dump.  
Every major transformation is accompanied by:
1. what is being done,
2. why it is being done,
3. what trade-off or source inconsistency it addresses.


## 2. Imports and runtime setup

This cell contains the small set of general-purpose libraries used throughout the finished notebook.  
The imports are intentionally compact: each dependency supports reproducible file handling, tabular transformation, or explicit validation/export behavior.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

## 3. Project paths and folder conventions

The notebook resolves the `part_2/` directory first, so its relative paths remain stable whether VS Code runs it from:
- the repository root, or
- the `part_2/` folder itself.

This matters for reruns: the same code should find the same raw workbooks and refresh the same processed output paths without manual path edits.

The raw-vs-processed rule is simple:
- `data/raw/` contains the unchanged MYH Excel inputs,
- `data/processed/` contains notebook-generated exports.


In [2]:
def resolve_part_2_dir() -> Path:
    """Return the Part 2 workspace when run from repo root or from part_2/."""
    cwd = Path.cwd().resolve()

    if cwd.name == "part_2":
        return cwd

    candidate = cwd / "part_2"
    if candidate.exists():
        return candidate

    raise FileNotFoundError(
        "Could not locate the 'part_2' folder. "
        "Run this notebook from the repository root or from the part_2/ folder."
    )


PART_2_DIR = resolve_part_2_dir()
RAW_DATA_DIR = PART_2_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PART_2_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Part 2 workspace: {PART_2_DIR}")
print(f"Raw input folder: {RAW_DATA_DIR}")
print(f"Processed export folder: {PROCESSED_DATA_DIR}")


Part 2 workspace: /mnt/data/dual_export_edit/complete/part_2
Raw input folder: /mnt/data/dual_export_edit/complete/part_2/data/raw
Processed export folder: /mnt/data/dual_export_edit/complete/part_2/data/processed


## 4. Raw-data inventory check

This early setup check confirms which Excel files are currently present in `data/raw/`.  
It acts as a practical fail-fast guard before deeper profiling and transformation begins: if a source workbook is missing or unexpected, the problem should appear near the top of the notebook rather than after later processing.


In [3]:
EXPECTED_SOURCE_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
SOURCE_YEAR_RE = re.compile(r"(20\d{2})")

raw_excel_files = sorted(RAW_DATA_DIR.glob("*.xlsx"))

print(f"Excel workbooks currently found: {len(raw_excel_files)}")
for file_path in raw_excel_files:
    print(f"- {file_path.name}")

if not raw_excel_files:
    print(
        "\nNo raw Excel files have been found yet. "
        "Place the six original MYH 2020–2025 workbooks in data/raw/ "
        "before starting the source exploration section."
    )

Excel workbooks currently found: 6
- resultat-ansokningsomgang-2020.xlsx
- resultat-ansokningsomgang-2021.xlsx
- resultat-ansokningsomgang-2022.xlsx
- resultat-ansokningsomgang-2023.xlsx
- resultat-ansokningsomgang-2024.xlsx
- resultat-ansokningsomgang-2025.xlsx


## 5. Source-file understanding

Before designing the curated target table, the notebook first profiles the source workbooks directly.  
This section is deliberately **exploratory but rerunnable**: it turns important source assumptions into visible evidence rather than relying on informal notes.

The profiling below answers six practical questions:
1. Which workbooks and sheets are present?
2. Where are the real headers located in the relevant source tables?
3. Does `Tabell 3` support the intended one-application-per-row main table?
4. Why must `Tabell 4` not be blindly merged into that main table?
5. How much does the `Tabell 3` schema change between 2020 and 2025?
6. Which source-value differences already foreshadow later harmonization work?

### 5.1 Workbook and sheet inventory

The source set is expected to contain one MYH workbook for each application round from **2020** through **2025**.  
This inventory checks both the file set and the internal worksheet layout.

In [4]:
def extract_source_year(file_path: Path) -> int:
    """Extract the MYH application-round year from a workbook filename."""
    match = SOURCE_YEAR_RE.search(file_path.name)
    if match is None:
        raise ValueError(f"Could not extract a source year from: {file_path.name}")
    return int(match.group(1))

source_workbooks = [
    {
        "source_year": extract_source_year(file_path),
        "source_file": file_path.name,
        "file_path": file_path,
    }
    for file_path in raw_excel_files
]

observed_source_years = sorted(workbook["source_year"] for workbook in source_workbooks)
missing_source_years = sorted(set(EXPECTED_SOURCE_YEARS) - set(observed_source_years))
unexpected_source_years = sorted(set(observed_source_years) - set(EXPECTED_SOURCE_YEARS))

if missing_source_years:
    raise FileNotFoundError(
        "Missing expected MYH source workbook(s) for year(s): "
        f"{missing_source_years}."
    )

if unexpected_source_years:
    print(
        "Note: additional Excel workbook years were found outside the planned 2020–2025 scope: "
        f"{unexpected_source_years}. They are visible in the inventory but are not part of the agreed Part 2 scope."
    )

workbook_inventory_rows = []
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    excel_file = pd.ExcelFile(workbook["file_path"])
    sheet_names = excel_file.sheet_names
    definition_sheet_name = next(
        (sheet for sheet in sheet_names if sheet.startswith("Definitioner")),
        None,
    )
    workbook_inventory_rows.append(
        {
            "source_year": workbook["source_year"],
            "source_file": workbook["source_file"],
            "sheet_count": len(sheet_names),
            "definition_sheet": definition_sheet_name,
            "sheet_names": " | ".join(sheet_names),
        }
    )

workbook_inventory = pd.DataFrame(workbook_inventory_rows).sort_values("source_year").reset_index(drop=True)

display(workbook_inventory)

,source_year,source_file,sheet_count,definition_sheet,sheet_names
0,2020,resultat-ansokningsomgang-2020.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
1,2021,resultat-ansokningsomgang-2021.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
2,2022,resultat-ansokningsomgang-2022.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
3,2023,resultat-ansokningsomgang-2023.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
4,2024,resultat-ansokningsomgang-2024.xlsx,6,Definitioner,Innehållsförteckning | Definitioner | Tabell 1...
5,2025,resultat-ansokningsomgang-2025.xlsx,6,Definitioner,Innehållsförteckning | Definitioner | Tabell 1...


### 5.2 Header-row detection for `Tabell 3` and `Tabell 4`

The relevant tables do **not** start on the same Excel row in every year.  
Instead of hard-coding the result as prose only, this profiling step scans the first rows of each table and locates the row containing `Diarienummer`, which is part of the true header row.

The output records both:
- the human-facing Excel row number, and
- the zero-based `header=` index that `pandas.read_excel()` would use later.

In [5]:
PROFILE_SHEETS = ["Tabell 3", "Tabell 4"]
HEADER_SCAN_ROWS = 20
HEADER_MARKER = "Diarienummer"


def detect_header_row(file_path: Path, sheet_name: str, marker: str = HEADER_MARKER) -> int:
    """Return the zero-based row index containing the table header marker."""
    preview = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None,
        nrows=HEADER_SCAN_ROWS,
    )

    matching_rows = []
    for row_index in range(len(preview)):
        row_values = {
            str(value).strip()
            for value in preview.iloc[row_index].tolist()
            if pd.notna(value)
        }
        if marker in row_values:
            matching_rows.append(row_index)

    if not matching_rows:
        raise ValueError(
            f"Could not find header marker '{marker}' in the first {HEADER_SCAN_ROWS} rows "
            f"of sheet '{sheet_name}' in '{file_path.name}'."
        )

    return matching_rows[0]


header_row_records = []
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    for sheet_name in PROFILE_SHEETS:
        header_index = detect_header_row(workbook["file_path"], sheet_name)
        header_row_records.append(
            {
                "source_year": workbook["source_year"],
                "source_sheet": sheet_name,
                "excel_header_row": header_index + 1,
                "pandas_header_index": header_index,
            }
        )

header_row_summary = pd.DataFrame(header_row_records).sort_values(
    ["source_sheet", "source_year"]
).reset_index(drop=True)

header_index_lookup = {
    (row.source_year, row.source_sheet): int(row.pandas_header_index)
    for row in header_row_summary.itertuples(index=False)
}

display(header_row_summary)

,source_year,source_sheet,excel_header_row,pandas_header_index
0,2020,Tabell 3,1,0
1,2021,Tabell 3,1,0
2,2022,Tabell 3,1,0
3,2023,Tabell 3,6,5
4,2024,Tabell 3,6,5
5,2025,Tabell 3,7,6
6,2020,Tabell 4,1,0
7,2021,Tabell 4,1,0
8,2022,Tabell 4,1,0
9,2023,Tabell 4,6,5


### 5.3 Read cleaned profiling copies of the source tables

The helper below is **not yet the production ingestion pipeline**.  
Its role in Sub-project 2.2 is narrower: read each profiled source table consistently enough to count rows, inspect identifiers, and compare source schemas.

In [6]:
def read_profile_table(file_path: Path, sheet_name: str, header_index: int) -> pd.DataFrame:
    """Read a source sheet for profiling and remove purely empty rows/columns."""
    table = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=header_index,
    )
    table = table.dropna(how="all").dropna(axis=1, how="all")
    table.columns = [str(column).strip() for column in table.columns]
    return table


tabell_3_tables = {}
tabell_4_tables = {}
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    year = workbook["source_year"]
    file_path = workbook["file_path"]
    tabell_3_tables[year] = read_profile_table(
        file_path,
        "Tabell 3",
        header_index_lookup[(year, "Tabell 3")],
    )
    tabell_4_tables[year] = read_profile_table(
        file_path,
        "Tabell 4",
        header_index_lookup[(year, "Tabell 4")],
    )

print(f"Profiling copies created for Tabell 3: {sorted(tabell_3_tables)}")
print(f"Profiling copies created for Tabell 4: {sorted(tabell_4_tables)}")

Profiling copies created for Tabell 3: [2020, 2021, 2022, 2023, 2024, 2025]
Profiling copies created for Tabell 4: [2020, 2021, 2022, 2023, 2024, 2025]


### 5.4 `Tabell 3` profile: evidence for the main applications table

The intended main table grain is:
> **one row = one application in one application round**.

For that to be credible, `Tabell 3` should have:
- a clear row count by year,
- no missing `Diarienummer`, and
- no duplicate `Diarienummer` within a year.

In [7]:
tabell_3_profile_rows = []
for year, table in sorted(tabell_3_tables.items()):
    identifiers = table["Diarienummer"]
    tabell_3_profile_rows.append(
        {
            "source_year": year,
            "tabell_3_rows": len(table),
            "tabell_3_columns": len(table.columns),
            "missing_diarienummer": int(identifiers.isna().sum()),
            "duplicate_diarienummer": int(identifiers.duplicated().sum()),
        }
    )

tabell_3_profile = pd.DataFrame(tabell_3_profile_rows).sort_values("source_year").reset_index(drop=True)

display(tabell_3_profile)

assert int(tabell_3_profile["missing_diarienummer"].sum()) == 0, "Unexpected missing Diarienummer in Tabell 3."
assert int(tabell_3_profile["duplicate_diarienummer"].sum()) == 0, "Unexpected duplicate Diarienummer in Tabell 3."

,source_year,tabell_3_rows,tabell_3_columns,missing_diarienummer,duplicate_diarienummer
0,2020,1482,16,0,0
1,2021,1238,17,0,0
2,2022,1207,16,0,0
3,2023,1258,28,0,0
4,2024,1272,28,0,0
5,2025,1184,28,0,0


### 5.5 `Tabell 4` profile: grain warning before any future joins

`Tabell 4` is useful, but it is **not** at the same grain as the planned main applications table.  
The check below compares total rows with the number of unique `Diarienummer` values.  A positive difference shows that at least some applications appear multiple times in `Tabell 4`.

In [8]:
tabell_4_grain_rows = []
for year, table in sorted(tabell_4_tables.items()):
    unique_application_ids = int(table["Diarienummer"].nunique(dropna=True))
    tabell_4_grain_rows.append(
        {
            "source_year": year,
            "tabell_4_rows": len(table),
            "unique_diarienummer": unique_application_ids,
            "rows_above_unique_application_count": len(table) - unique_application_ids,
        }
    )

tabell_4_grain_check = pd.DataFrame(tabell_4_grain_rows).sort_values("source_year").reset_index(drop=True)

display(tabell_4_grain_check)

assert (
    tabell_4_grain_check["rows_above_unique_application_count"] > 0
).all(), "Expected Tabell 4 to show repeated application identifiers in every profiled year."

,source_year,tabell_4_rows,unique_diarienummer,rows_above_unique_application_count
0,2020,1903,1482,421
1,2021,1586,1238,348
2,2022,1621,1207,414
3,2023,642,243,399
4,2024,842,292,550
5,2025,836,295,541


### 5.6 `Tabell 3` schema comparison across 2020–2025

A cross-year curated dataset cannot assume that every source year has the same input structure.  
This comparison records:
- the column count by year, and
- which source columns appear in which years.

The result gives concrete evidence for the later harmonization/specification work in Sub-project 2.3.

In [9]:
tabell_3_schema_by_year = {
    year: list(table.columns)
    for year, table in sorted(tabell_3_tables.items())
}

schema_count_summary = pd.DataFrame(
    [
        {
            "source_year": year,
            "tabell_3_columns": len(columns),
            "column_names": " | ".join(columns),
        }
        for year, columns in tabell_3_schema_by_year.items()
    ]
).sort_values("source_year").reset_index(drop=True)

all_tabell_3_columns = sorted(
    set().union(*(set(columns) for columns in tabell_3_schema_by_year.values()))
)

schema_presence_rows = []
for column_name in all_tabell_3_columns:
    years_present = [
        year
        for year, columns in tabell_3_schema_by_year.items()
        if column_name in columns
    ]
    schema_presence_rows.append(
        {
            "column_name": column_name,
            "years_present": ", ".join(str(year) for year in years_present),
            "present_in_year_count": len(years_present),
        }
    )

schema_presence_summary = pd.DataFrame(schema_presence_rows).sort_values(
    ["present_in_year_count", "column_name"],
    ascending=[False, True],
).reset_index(drop=True)

schema_variation_summary = schema_presence_summary[
    schema_presence_summary["present_in_year_count"] < len(EXPECTED_SOURCE_YEARS)
].reset_index(drop=True)

display(schema_count_summary)
display(schema_variation_summary)

,source_year,tabell_3_columns,column_names
0,2020,16,Utbildningsområde | Utbildningsnamn | Län | Ko...
1,2021,17,Utbildningsområde | Utbildningsnamn | Län | Ko...
2,2022,16,Utbildningsområde | Utbildningsnamn | Beslut |...
3,2023,28,Utbildningsområde | SUN5 inriktning | SUN5 inr...
4,2024,28,Utbildningsområde | SUN5 inriktning | SUN5 inr...
5,2025,28,Utbildningsområde | SUN5 inriktning | SUN5 inr...


,column_name,years_present,present_in_year_count
0,Flera kommuner,"2022, 2023, 2024, 2025",4
1,Typ av examen,"2021, 2022, 2023, 2024",4
2,Beviljade platser totalt,"2023, 2024, 2025",3
3,Beviljade platser utbildningsomgång 1,"2023, 2024, 2025",3
4,Beviljade platser utbildningsomgång 2,"2023, 2024, 2025",3
5,Beviljade platser utbildningsomgång 3,"2023, 2024, 2025",3
6,Beviljade platser utbildningsomgång 4,"2023, 2024, 2025",3
7,Beviljade platser utbildningsomgång 5,"2023, 2024, 2025",3
8,SUN5 inriktning,"2023, 2024, 2025",3
9,SUN5 inriktning namn,"2023, 2024, 2025",3


### 5.7 Source values that already signal later harmonization needs

This is still source exploration, not cleaning.  
However, two columns already show year-to-year vocabulary differences that will matter later:
- `Beslut`, where rejection wording changes and 2025 adds `Återkallad`,
- `Huvudmannatyp`, where `Landsting` is replaced by `Region` in later years.

In [10]:
def summarize_distinct_values(tables_by_year: dict[int, pd.DataFrame], column_name: str) -> pd.DataFrame:
    """Summarize sorted distinct non-null source values for one column by source year."""
    rows = []
    for year, table in sorted(tables_by_year.items()):
        distinct_values = sorted(str(value) for value in table[column_name].dropna().unique())
        rows.append(
            {
                "source_year": year,
                "source_column": column_name,
                "distinct_values": " | ".join(distinct_values),
            }
        )
    return pd.DataFrame(rows)

beslut_value_summary = summarize_distinct_values(tabell_3_tables, "Beslut")
huvudmannatyp_value_summary = summarize_distinct_values(tabell_3_tables, "Huvudmannatyp")

display(beslut_value_summary)
display(huvudmannatyp_value_summary)

,source_year,source_column,distinct_values
0,2020,Beslut,Beviljad | Ej beviljad
1,2021,Beslut,Beviljad | Ej beviljad
2,2022,Beslut,Avslag | Beviljad
3,2023,Beslut,Avslag | Beviljad
4,2024,Beslut,Avslag | Beviljad
5,2025,Beslut,Avslag | Beviljad | Återkallad


,source_year,source_column,distinct_values
0,2020,Huvudmannatyp,Kommun | Landsting | Privat | Statlig
1,2021,Huvudmannatyp,Kommun | Landsting | Privat | Statlig
2,2022,Huvudmannatyp,Kommun | Privat | Region | Statlig
3,2023,Huvudmannatyp,Kommun | Privat | Region | Statlig
4,2024,Huvudmannatyp,Kommun | Privat | Region | Statlig
5,2025,Huvudmannatyp,Kommun | Privat | Region


## 6. Why `Tabell 3` is the main source and `Tabell 4` is not merged

The profiling evidence above supports the project’s current data-grain decision:

1. **`Tabell 3` fits the planned main table grain.**  
   Across all six source years, `Tabell 3` has one row per recorded application, zero missing `Diarienummer`, and zero duplicate `Diarienummer` values within each year.

2. **`Tabell 4` is at a more detailed grain.**  
   In every year, `Tabell 4` has more rows than unique `Diarienummer` values. That means some applications appear multiple times there.

3. **Therefore, `Tabell 4` must not be blindly joined into the main applications table.**  
   A direct merge would risk duplicating applications and corrupting counts. If `Tabell 4` is ever used later, it should be handled as a separate more-detailed table or joined only after an explicit aggregation/design decision.

This is a **grain conclusion**, not the full schema contract. The next section turns it into a curated target design and explicit source-to-target harmonization rules.


## 7. Target schema and harmonization specification

The source profile and grain decision are now converted into a formal curated-table contract before production ingestion begins.

### 7.0 Schema-design decisions now fixed
- The curated dataset remains a **single `Tabell 3`-based applications table**.
- The grain remains: **one row = one application in one application round**.
- Curated field names use lowercase `snake_case` without Swedish diacritics where practical.
- Raw categorical fields are retained when useful, and normalized companion fields are added where cross-year harmonization is required.
- Fields introduced only in later workbooks are **kept when they add clear analytical value**, and they will be null for earlier years where the source column does not exist.
- Source traceability is preserved through:
  - `source_year`
  - `source_file`
  - `source_sheet`
  - `source_row`

### 7.0.1 Traceability semantics
The final curated table will use the following meanings:

| Curated field | Design rule |
|---|---|
| `source_year` | MYH application-round year extracted from the workbook filename |
| `source_file` | Source workbook filename only, not an absolute local path |
| `source_sheet` | Expected to be `Tabell 3` for the main table |
| `source_row` | **1-based Excel row number** of the original source record in `Tabell 3` |

`source_row` is intentionally defined as an Excel row reference rather than a temporary DataFrame index.  
The later ingestion implementation can calculate it from the detected header position and row offset, preserving a direct audit path back to the raw workbook.

### 7.0.2 Structural-null policy
Some fields are valuable enough to retain even though they appear only in later source years.  
For those fields, earlier years will carry **structural nulls** during ingestion:

- not fabricated values,
- not silent zeroes,
- not treated as accidental data loss.

This policy keeps one stable target schema across 2020–2025 while preserving the historical limits of the raw sources.


In [11]:
CURATED_SCHEMA_FIELDS = [
    {
        "column_order": 1,
        "curated_field": "source_year",
        "field_group": "traceability",
        "source_basis": "workbook filename / import metadata",
        "availability": "all years",
        "implementation_note": "MYH application-round year",
    },
    {
        "column_order": 2,
        "curated_field": "source_file",
        "field_group": "traceability",
        "source_basis": "workbook filename",
        "availability": "all years",
        "implementation_note": "Portable filename only, not an absolute path",
    },
    {
        "column_order": 3,
        "curated_field": "source_sheet",
        "field_group": "traceability",
        "source_basis": "import metadata",
        "availability": "all years",
        "implementation_note": "Expected to be Tabell 3",
    },
    {
        "column_order": 4,
        "curated_field": "source_row",
        "field_group": "traceability",
        "source_basis": "detected header row + source-record offset",
        "availability": "all years",
        "implementation_note": "1-based Excel row number of the original Tabell 3 record",
    },
    {
        "column_order": 5,
        "curated_field": "diarienummer",
        "field_group": "application_identity",
        "source_basis": "Diarienummer",
        "availability": "all years",
        "implementation_note": "Main application identifier within source year",
    },
    {
        "column_order": 6,
        "curated_field": "utbildningsnamn",
        "field_group": "application_identity",
        "source_basis": "Utbildningsnamn",
        "availability": "all years",
        "implementation_note": "Program name",
    },
    {
        "column_order": 7,
        "curated_field": "utbildningsomrade",
        "field_group": "application_identity",
        "source_basis": "Utbildningsområde",
        "availability": "all years",
        "implementation_note": "Broad education area",
    },
    {
        "column_order": 8,
        "curated_field": "beslut",
        "field_group": "decision",
        "source_basis": "Beslut",
        "availability": "all years",
        "implementation_note": "Original source decision value retained",
    },
    {
        "column_order": 9,
        "curated_field": "beslut_normalized",
        "field_group": "decision",
        "source_basis": "derived from beslut",
        "availability": "all years",
        "implementation_note": "Cross-year decision category",
    },
    {
        "column_order": 10,
        "curated_field": "is_approved",
        "field_group": "decision",
        "source_basis": "derived from beslut_normalized",
        "availability": "all years",
        "implementation_note": "True only when beslut_normalized == 'approved'",
    },
    {
        "column_order": 11,
        "curated_field": "lan",
        "field_group": "geography",
        "source_basis": "Län",
        "availability": "all years",
        "implementation_note": "County recorded in Tabell 3",
    },
    {
        "column_order": 12,
        "curated_field": "kommun",
        "field_group": "geography",
        "source_basis": "Kommun",
        "availability": "all years",
        "implementation_note": "Municipality recorded in Tabell 3",
    },
    {
        "column_order": 13,
        "curated_field": "flera_kommuner",
        "field_group": "geography",
        "source_basis": "Flera studiekommuner / Flera kommuner",
        "availability": "all years",
        "implementation_note": "Source yes/no indicator under a harmonized target name",
    },
    {
        "column_order": 14,
        "curated_field": "has_multiple_municipalities",
        "field_group": "geography",
        "source_basis": "derived from flera_kommuner",
        "availability": "all years",
        "implementation_note": "Boolean convenience field; Ja -> True, Nej -> False",
    },
    {
        "column_order": 15,
        "curated_field": "antal_kommuner",
        "field_group": "geography",
        "source_basis": "Antal kommuner",
        "availability": "all years",
        "implementation_note": "Number of municipalities",
    },
    {
        "column_order": 16,
        "curated_field": "yh_poang",
        "field_group": "program_structure",
        "source_basis": "YH-poäng",
        "availability": "all years",
        "implementation_note": "Program extent in YH credits",
    },
    {
        "column_order": 17,
        "curated_field": "studieform",
        "field_group": "program_structure",
        "source_basis": "Studieform",
        "availability": "all years",
        "implementation_note": "Original source delivery form",
    },
    {
        "column_order": 18,
        "curated_field": "is_distance_based",
        "field_group": "program_structure",
        "source_basis": "derived from studieform",
        "availability": "all years",
        "implementation_note": "Boolean convenience field; Distans -> True, Bunden -> False",
    },
    {
        "column_order": 19,
        "curated_field": "studietakt_procent",
        "field_group": "program_structure",
        "source_basis": "Studietakt %",
        "availability": "all years",
        "implementation_note": "Study pace as a percentage",
    },
    {
        "column_order": 20,
        "curated_field": "examenstyp",
        "field_group": "program_structure",
        "source_basis": "Typ av examen / Examenstyp",
        "availability": "2021-2025; structural null in 2020",
        "implementation_note": "Harmonized target name for the exam-type field",
    },
    {
        "column_order": 21,
        "curated_field": "utbildningsanordnare",
        "field_group": "provider",
        "source_basis": "Utbildningsanordnare administrativ enhet",
        "availability": "all years",
        "implementation_note": "Provider / administrative unit",
    },
    {
        "column_order": 22,
        "curated_field": "huvudmannatyp",
        "field_group": "provider",
        "source_basis": "Huvudmannatyp",
        "availability": "all years",
        "implementation_note": "Original source provider-type value retained",
    },
    {
        "column_order": 23,
        "curated_field": "huvudmannatyp_normalized",
        "field_group": "provider",
        "source_basis": "derived from huvudmannatyp",
        "availability": "all years",
        "implementation_note": "Landsting and Region harmonized to Region",
    },
    {
        "column_order": 24,
        "curated_field": "sokta_utbildningsomgangar",
        "field_group": "application_scope",
        "source_basis": "Sökta utbildningsomgångar",
        "availability": "all years",
        "implementation_note": "Requested education rounds",
    },
    {
        "column_order": 25,
        "curated_field": "beviljade_utbildningsomgangar",
        "field_group": "application_scope",
        "source_basis": "Beviljade utbildningsomgångar",
        "availability": "all years",
        "implementation_note": "Approved education rounds",
    },
    {
        "column_order": 26,
        "curated_field": "sun5_inriktning",
        "field_group": "newer_classification",
        "source_basis": "SUN5 inriktning",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained because it adds useful classification detail",
    },
    {
        "column_order": 27,
        "curated_field": "sun5_inriktning_namn",
        "field_group": "newer_classification",
        "source_basis": "SUN5 inriktning namn",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Human-readable SUN5 label",
    },
    {
        "column_order": 28,
        "curated_field": "seqf_niva",
        "field_group": "newer_classification",
        "source_basis": "SeQF nivå",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a later-year classification field",
    },
    {
        "column_order": 29,
        "curated_field": "smalt_yrkesomrade",
        "field_group": "newer_classification",
        "source_basis": "Smalt yrkesområde",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a later-year classification field",
    },
    {
        "column_order": 30,
        "curated_field": "sokta_platser_per_utbildningsomgang",
        "field_group": "newer_seat_totals",
        "source_basis": "Sökta platser per utbildningsomgång",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a compact capacity-related summary",
    },
    {
        "column_order": 31,
        "curated_field": "sokta_platser_totalt",
        "field_group": "newer_seat_totals",
        "source_basis": "Sökta platser totalt",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a compact capacity-related summary",
    },
    {
        "column_order": 32,
        "curated_field": "beviljade_platser_totalt",
        "field_group": "newer_seat_totals",
        "source_basis": "Beviljade platser totalt",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a compact capacity-related summary",
    },
]

curated_schema_spec = pd.DataFrame(CURATED_SCHEMA_FIELDS).sort_values("column_order").reset_index(drop=True)

display(curated_schema_spec)

,column_order,curated_field,field_group,source_basis,availability,implementation_note
0,1,source_year,traceability,workbook filename / import metadata,all years,MYH application-round year
1,2,source_file,traceability,workbook filename,all years,"Portable filename only, not an absolute path"
2,3,source_sheet,traceability,import metadata,all years,Expected to be Tabell 3
3,4,source_row,traceability,detected header row + source-record offset,all years,1-based Excel row number of the original Tabel...
4,5,diarienummer,application_identity,Diarienummer,all years,Main application identifier within source year
5,6,utbildningsnamn,application_identity,Utbildningsnamn,all years,Program name
6,7,utbildningsomrade,application_identity,Utbildningsområde,all years,Broad education area
7,8,beslut,decision,Beslut,all years,Original source decision value retained
8,9,beslut_normalized,decision,derived from beslut,all years,Cross-year decision category
9,10,is_approved,decision,derived from beslut_normalized,all years,True only when beslut_normalized == 'approved'


### 7.1 Source-to-target mapping table

The table below formalizes how the final curated fields connect back to the observed `Tabell 3` source columns.  
It is intentionally a **design specification**, not yet the reusable ingestion pipeline. The production functions that apply these rules belong to Sub-project **2.4**.


In [12]:
SOURCE_TO_TARGET_MAPPING = [
    {"curated_field": "source_year", "source_column_or_rule": "import metadata from workbook filename", "source_years": "2020-2025", "harmonization_rule": "Extract the application-round year"},
    {"curated_field": "source_file", "source_column_or_rule": "workbook filename", "source_years": "2020-2025", "harmonization_rule": "Store filename only"},
    {"curated_field": "source_sheet", "source_column_or_rule": "sheet metadata", "source_years": "2020-2025", "harmonization_rule": "Store 'Tabell 3'"},
    {"curated_field": "source_row", "source_column_or_rule": "detected Excel source row", "source_years": "2020-2025", "harmonization_rule": "Store 1-based Excel row number"},
    {"curated_field": "diarienummer", "source_column_or_rule": "Diarienummer", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "utbildningsnamn", "source_column_or_rule": "Utbildningsnamn", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "utbildningsomrade", "source_column_or_rule": "Utbildningsområde", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "beslut", "source_column_or_rule": "Beslut", "source_years": "2020-2025", "harmonization_rule": "Retain original source value"},
    {"curated_field": "beslut_normalized", "source_column_or_rule": "derived from beslut", "source_years": "2020-2025", "harmonization_rule": "Beviljad -> approved; Ej beviljad/Avslag -> rejected; Återkallad -> withdrawn"},
    {"curated_field": "is_approved", "source_column_or_rule": "derived from beslut_normalized", "source_years": "2020-2025", "harmonization_rule": "approved -> True; otherwise False"},
    {"curated_field": "lan", "source_column_or_rule": "Län", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "kommun", "source_column_or_rule": "Kommun", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "flera_kommuner", "source_column_or_rule": "2020-2021: Flera studiekommuner; 2022-2025: Flera kommuner", "source_years": "2020-2025", "harmonization_rule": "Map both source names into one target field"},
    {"curated_field": "has_multiple_municipalities", "source_column_or_rule": "derived from flera_kommuner", "source_years": "2020-2025", "harmonization_rule": "Ja -> True; Nej -> False"},
    {"curated_field": "antal_kommuner", "source_column_or_rule": "Antal kommuner", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "yh_poang", "source_column_or_rule": "YH-poäng", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "studieform", "source_column_or_rule": "Studieform", "source_years": "2020-2025", "harmonization_rule": "Retain original source value"},
    {"curated_field": "is_distance_based", "source_column_or_rule": "derived from studieform", "source_years": "2020-2025", "harmonization_rule": "Distans -> True; Bunden -> False"},
    {"curated_field": "studietakt_procent", "source_column_or_rule": "Studietakt %", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "examenstyp", "source_column_or_rule": "2021-2024: Typ av examen; 2025: Examenstyp", "source_years": "2021-2025", "harmonization_rule": "Map both source names into one target field; structural null in 2020"},
    {"curated_field": "utbildningsanordnare", "source_column_or_rule": "Utbildningsanordnare administrativ enhet", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "huvudmannatyp", "source_column_or_rule": "Huvudmannatyp", "source_years": "2020-2025", "harmonization_rule": "Retain original source value"},
    {"curated_field": "huvudmannatyp_normalized", "source_column_or_rule": "derived from huvudmannatyp", "source_years": "2020-2025", "harmonization_rule": "Landsting/Region -> Region; other categories retained"},
    {"curated_field": "sokta_utbildningsomgangar", "source_column_or_rule": "Sökta utbildningsomgångar", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "beviljade_utbildningsomgangar", "source_column_or_rule": "Beviljade utbildningsomgångar", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "sun5_inriktning", "source_column_or_rule": "SUN5 inriktning", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "sun5_inriktning_namn", "source_column_or_rule": "SUN5 inriktning namn", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "seqf_niva", "source_column_or_rule": "SeQF nivå", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "smalt_yrkesomrade", "source_column_or_rule": "Smalt yrkesområde", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "sokta_platser_per_utbildningsomgang", "source_column_or_rule": "Sökta platser per utbildningsomgång", "source_years": "2023-2025", "harmonization_rule": "Retain summary field; structural null in 2020-2022"},
    {"curated_field": "sokta_platser_totalt", "source_column_or_rule": "Sökta platser totalt", "source_years": "2023-2025", "harmonization_rule": "Retain summary field; structural null in 2020-2022"},
    {"curated_field": "beviljade_platser_totalt", "source_column_or_rule": "Beviljade platser totalt", "source_years": "2023-2025", "harmonization_rule": "Retain summary field; structural null in 2020-2022"},
]

source_to_target_mapping = pd.DataFrame(SOURCE_TO_TARGET_MAPPING)

assert list(source_to_target_mapping["curated_field"]) == list(curated_schema_spec["curated_field"]), (
    "The source-to-target mapping should match the locked curated schema order."
)

display(source_to_target_mapping)

,curated_field,source_column_or_rule,source_years,harmonization_rule
0,source_year,import metadata from workbook filename,2020-2025,Extract the application-round year
1,source_file,workbook filename,2020-2025,Store filename only
2,source_sheet,sheet metadata,2020-2025,Store 'Tabell 3'
3,source_row,detected Excel source row,2020-2025,Store 1-based Excel row number
4,diarienummer,Diarienummer,2020-2025,Rename only
5,utbildningsnamn,Utbildningsnamn,2020-2025,Rename only
6,utbildningsomrade,Utbildningsområde,2020-2025,Rename only
7,beslut,Beslut,2020-2025,Retain original source value
8,beslut_normalized,derived from beslut,2020-2025,Beviljad -> approved; Ej beviljad/Avslag -> re...
9,is_approved,derived from beslut_normalized,2020-2025,approved -> True; otherwise False


### 7.2 Inclusion and exclusion decisions for observed `Tabell 3` columns

A good target schema should not silently drop source fields or silently include every field without a reason.  
The table below accounts for **every observed `Tabell 3` source column** and records whether it is:
- retained directly,
- harmonized into a target field,
- or excluded from the main curated applications table.


In [13]:
SOURCE_COLUMN_DISPOSITION = [
    {"source_column": "Utbildningsområde", "design_status": "include", "curated_target": "utbildningsomrade", "reason": "Core cross-year program descriptor"},
    {"source_column": "Utbildningsnamn", "design_status": "include", "curated_target": "utbildningsnamn", "reason": "Core application/program descriptor"},
    {"source_column": "Län", "design_status": "include", "curated_target": "lan", "reason": "Core geography field"},
    {"source_column": "Kommun", "design_status": "include", "curated_target": "kommun", "reason": "Core geography field"},
    {"source_column": "Flera studiekommuner", "design_status": "harmonize", "curated_target": "flera_kommuner", "reason": "Older source name for the same multi-municipality concept"},
    {"source_column": "Flera kommuner", "design_status": "harmonize", "curated_target": "flera_kommuner", "reason": "Later source name for the same multi-municipality concept"},
    {"source_column": "Antal kommuner", "design_status": "include", "curated_target": "antal_kommuner", "reason": "Useful count field with stable meaning"},
    {"source_column": "Antal län", "design_status": "exclude", "curated_target": "", "reason": "Only present in 2020 and not needed for the chosen main-table design"},
    {"source_column": "YH-poäng", "design_status": "include", "curated_target": "yh_poang", "reason": "Core program-structure measure"},
    {"source_column": "Studieform", "design_status": "include", "curated_target": "studieform", "reason": "Core program-structure category"},
    {"source_column": "Studietakt %", "design_status": "include", "curated_target": "studietakt_procent", "reason": "Core program-structure measure"},
    {"source_column": "Typ av examen", "design_status": "harmonize", "curated_target": "examenstyp", "reason": "2021-2024 source name for exam type"},
    {"source_column": "Examenstyp", "design_status": "harmonize", "curated_target": "examenstyp", "reason": "2025 source name for exam type"},
    {"source_column": "Utbildningsanordnare administrativ enhet", "design_status": "include", "curated_target": "utbildningsanordnare", "reason": "Core provider descriptor"},
    {"source_column": "Huvudmannatyp", "design_status": "include", "curated_target": "huvudmannatyp", "reason": "Raw provider-type category retained before normalization"},
    {"source_column": "Sökta utbildningsomgångar", "design_status": "include", "curated_target": "sokta_utbildningsomgangar", "reason": "Cross-year application-scope field"},
    {"source_column": "Beviljade utbildningsomgångar", "design_status": "include", "curated_target": "beviljade_utbildningsomgangar", "reason": "Cross-year application-scope field"},
    {"source_column": "Diarienummer", "design_status": "include", "curated_target": "diarienummer", "reason": "Application identifier"},
    {"source_column": "Beslut", "design_status": "include", "curated_target": "beslut", "reason": "Raw decision value retained before normalization"},
    {"source_column": "SUN5 inriktning", "design_status": "include", "curated_target": "sun5_inriktning", "reason": "Useful 2023-2025 classification field; earlier years structurally null"},
    {"source_column": "SUN5 inriktning namn", "design_status": "include", "curated_target": "sun5_inriktning_namn", "reason": "Useful 2023-2025 classification label; earlier years structurally null"},
    {"source_column": "SeQF nivå", "design_status": "include", "curated_target": "seqf_niva", "reason": "Useful 2023-2025 classification field; earlier years structurally null"},
    {"source_column": "Smalt yrkesområde", "design_status": "include", "curated_target": "smalt_yrkesomrade", "reason": "Useful 2023-2025 classification field; earlier years structurally null"},
    {"source_column": "Sökta platser per utbildningsomgång", "design_status": "include", "curated_target": "sokta_platser_per_utbildningsomgang", "reason": "Compact 2023-2025 seat-demand summary; earlier years structurally null"},
    {"source_column": "Sökta platser totalt", "design_status": "include", "curated_target": "sokta_platser_totalt", "reason": "Compact 2023-2025 seat-demand summary; earlier years structurally null"},
    {"source_column": "Beviljade platser totalt", "design_status": "include", "curated_target": "beviljade_platser_totalt", "reason": "Compact 2023-2025 seat-approval summary; earlier years structurally null"},
    {"source_column": "Beviljade platser utbildningsomgång 1", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 2", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 3", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 4", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 5", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
]

source_column_disposition = pd.DataFrame(SOURCE_COLUMN_DISPOSITION).sort_values(
    ["design_status", "source_column"]
).reset_index(drop=True)

observed_tabell_3_source_columns = set(all_tabell_3_columns)
accounted_source_columns = set(source_column_disposition["source_column"])

assert accounted_source_columns == observed_tabell_3_source_columns, (
    "Every observed Tabell 3 source column must be explicitly accounted for in the design disposition."
)

display(source_column_disposition)

,source_column,design_status,curated_target,reason
0,Antal län,exclude,,Only present in 2020 and not needed for the ch...
1,Beviljade platser utbildningsomgång 1,exclude,,Round-specific repeating measure; too wide/spa...
2,Beviljade platser utbildningsomgång 2,exclude,,Round-specific repeating measure; too wide/spa...
3,Beviljade platser utbildningsomgång 3,exclude,,Round-specific repeating measure; too wide/spa...
4,Beviljade platser utbildningsomgång 4,exclude,,Round-specific repeating measure; too wide/spa...
5,Beviljade platser utbildningsomgång 5,exclude,,Round-specific repeating measure; too wide/spa...
6,Examenstyp,harmonize,examenstyp,2025 source name for exam type
7,Flera kommuner,harmonize,flera_kommuner,Later source name for the same multi-municipal...
8,Flera studiekommuner,harmonize,flera_kommuner,Older source name for the same multi-municipal...
9,Typ av examen,harmonize,examenstyp,2021-2024 source name for exam type


### 7.3 Normalized-value specifications and design-level coverage checks

Two cross-year categorical harmonizations are formally locked here:

1. `Beslut` → `beslut_normalized`
2. `Huvudmannatyp` → `huvudmannatyp_normalized`

The mappings are validated against the distinct source values already observed in the six profiled workbooks.  
This remains a **design-level coverage check**; the reusable transformation code will be implemented in Sub-project **2.4**.


In [14]:
BESLUT_NORMALIZATION = {
    "Beviljad": "approved",
    "Ej beviljad": "rejected",
    "Avslag": "rejected",
    "Återkallad": "withdrawn",
}

HUVUDMANNATYP_NORMALIZATION = {
    "Landsting": "Region",
    "Region": "Region",
    "Kommun": "Kommun",
    "Privat": "Privat",
    "Statlig": "Statlig",
}

beslut_normalization_spec = pd.DataFrame(
    [
        {"source_value": source_value, "normalized_value": normalized_value}
        for source_value, normalized_value in BESLUT_NORMALIZATION.items()
    ]
).sort_values("source_value").reset_index(drop=True)

huvudmannatyp_normalization_spec = pd.DataFrame(
    [
        {"source_value": source_value, "normalized_value": normalized_value}
        for source_value, normalized_value in HUVUDMANNATYP_NORMALIZATION.items()
    ]
).sort_values("source_value").reset_index(drop=True)

observed_beslut_values = {
    str(value).strip()
    for table in tabell_3_tables.values()
    for value in table["Beslut"].dropna().unique()
}
observed_huvudmannatyp_values = {
    str(value).strip()
    for table in tabell_3_tables.values()
    for value in table["Huvudmannatyp"].dropna().unique()
}

unmapped_beslut_values = sorted(observed_beslut_values - set(BESLUT_NORMALIZATION))
unmapped_huvudmannatyp_values = sorted(
    observed_huvudmannatyp_values - set(HUVUDMANNATYP_NORMALIZATION)
)

normalization_coverage_summary = pd.DataFrame(
    [
        {
            "field": "Beslut -> beslut_normalized",
            "observed_source_values": " | ".join(sorted(observed_beslut_values)),
            "unmapped_source_values": " | ".join(unmapped_beslut_values) if unmapped_beslut_values else "None",
        },
        {
            "field": "Huvudmannatyp -> huvudmannatyp_normalized",
            "observed_source_values": " | ".join(sorted(observed_huvudmannatyp_values)),
            "unmapped_source_values": " | ".join(unmapped_huvudmannatyp_values) if unmapped_huvudmannatyp_values else "None",
        },
    ]
)

display(beslut_normalization_spec)
display(huvudmannatyp_normalization_spec)
display(normalization_coverage_summary)

assert not unmapped_beslut_values, "Observed Beslut value(s) are missing from BESLUT_NORMALIZATION."
assert not unmapped_huvudmannatyp_values, "Observed Huvudmannatyp value(s) are missing from HUVUDMANNATYP_NORMALIZATION."

,source_value,normalized_value
0,Avslag,rejected
1,Beviljad,approved
2,Ej beviljad,rejected
3,Återkallad,withdrawn


,source_value,normalized_value
0,Kommun,Kommun
1,Landsting,Region
2,Privat,Privat
3,Region,Region
4,Statlig,Statlig


,field,observed_source_values,unmapped_source_values
0,Beslut -> beslut_normalized,Avslag | Beviljad | Ej beviljad | Återkallad,None
1,Huvudmannatyp -> huvudmannatyp_normalized,Kommun | Landsting | Privat | Region | Statlig,None


### 7.4 Schema conclusion

The target design is now sufficiently specific for implementation:

- the **final curated field set and order** are fixed,
- each target field has a documented source basis,
- every observed `Tabell 3` source column has an explicit keep / harmonize / exclude decision,
- later-year fields have an explicit structural-null policy,
- and the two main categorical normalization rules already have design-level coverage checks.

The next section converts this specification into a reusable ingestion-and-standardization pipeline that creates consistent, year-specific intermediate tables before later cleaning and enrichment.


## 8. Reusable ingestion and standardization *(Sub-project 2.4)*

With the target schema fixed, the notebook turns the design into reusable import logic. This section keeps import mechanics separate from later cleaning and enrichment: first record the year-specific `Tabell 3` layout, then use one reader function instead of repeated ad hoc `read_excel(...)` calls, and then standardize/concatenate in the next subsection.

### 8.1 Year-aware `Tabell 3` import configuration and reusable reader

The real `Tabell 3` header row differs across the six MYH workbooks. A compact configuration table makes that structural difference explicit, reviewable, and rerunnable. The reader below uses the configuration, removes purely empty rows/columns, and preserves portable traceability fields, including the original 1-based Excel source row.


In [15]:
TABELL_3_HEADER_ROWS = {
    2020: {"excel_header_row": 1, "pandas_header_index": 0},
    2021: {"excel_header_row": 1, "pandas_header_index": 0},
    2022: {"excel_header_row": 1, "pandas_header_index": 0},
    2023: {"excel_header_row": 6, "pandas_header_index": 5},
    2024: {"excel_header_row": 6, "pandas_header_index": 5},
    2025: {"excel_header_row": 7, "pandas_header_index": 6},
}

source_workbook_by_year = {
    int(workbook["source_year"]): workbook
    for workbook in source_workbooks
}

source_import_config = pd.DataFrame(
    [
        {
            "source_year": year,
            "source_file": source_workbook_by_year[year]["source_file"],
            "source_sheet": "Tabell 3",
            "excel_header_row": header_metadata["excel_header_row"],
            "pandas_header_index": header_metadata["pandas_header_index"],
        }
        for year, header_metadata in TABELL_3_HEADER_ROWS.items()
    ]
).sort_values("source_year").reset_index(drop=True)

assert list(source_import_config["source_year"]) == EXPECTED_SOURCE_YEARS, (
    "The Tabell 3 source import configuration must cover the agreed 2020–2025 scope."
)
assert source_import_config["source_file"].is_unique, (
    "Each configured source year should resolve to one portable workbook filename."
)

display(source_import_config)

LOCKED_CURATED_SCHEMA = list(curated_schema_spec["curated_field"])

SOURCE_TO_STANDARDIZED_COLUMN = {
    "Diarienummer": "diarienummer",
    "Utbildningsnamn": "utbildningsnamn",
    "Utbildningsområde": "utbildningsomrade",
    "Beslut": "beslut",
    "Län": "lan",
    "Kommun": "kommun",
    "Flera studiekommuner": "flera_kommuner",
    "Flera kommuner": "flera_kommuner",
    "Antal kommuner": "antal_kommuner",
    "YH-poäng": "yh_poang",
    "Studieform": "studieform",
    "Studietakt %": "studietakt_procent",
    "Typ av examen": "examenstyp",
    "Examenstyp": "examenstyp",
    "Utbildningsanordnare administrativ enhet": "utbildningsanordnare",
    "Huvudmannatyp": "huvudmannatyp",
    "Sökta utbildningsomgångar": "sokta_utbildningsomgangar",
    "Beviljade utbildningsomgångar": "beviljade_utbildningsomgangar",
    "SUN5 inriktning": "sun5_inriktning",
    "SUN5 inriktning namn": "sun5_inriktning_namn",
    "SeQF nivå": "seqf_niva",
    "Smalt yrkesområde": "smalt_yrkesomrade",
    "Sökta platser per utbildningsomgång": "sokta_platser_per_utbildningsomgang",
    "Sökta platser totalt": "sokta_platser_totalt",
    "Beviljade platser totalt": "beviljade_platser_totalt",
}

INGESTION_STAGE_PLACEHOLDER_FIELDS = [
    "beslut_normalized",
    "is_approved",
    "has_multiple_municipalities",
    "is_distance_based",
    "huvudmannatyp_normalized",
]

CONCAT_STABLE_NULLABLE_INTEGER_FIELDS = [
    "seqf_niva",
    "sokta_platser_per_utbildningsomgang",
    "sokta_platser_totalt",
    "beviljade_platser_totalt",
]


def read_standardized_tabell_3(config: dict[str, object]) -> pd.DataFrame:
    """Read one configured Tabell 3 sheet and align it to the locked 32-field schema."""
    file_path = RAW_DATA_DIR / str(config["source_file"])
    source_table = pd.read_excel(
        file_path,
        sheet_name=str(config["source_sheet"]),
        header=int(config["pandas_header_index"]),
    )
    source_table.columns = [str(column).strip() for column in source_table.columns]

    source_data_columns = list(source_table.columns)
    source_table["_source_row"] = np.arange(
        int(config["excel_header_row"]) + 1,
        int(config["excel_header_row"]) + 1 + len(source_table),
    )

    non_empty_row_mask = source_table[source_data_columns].notna().any(axis=1)
    source_table = source_table.loc[non_empty_row_mask].copy()
    source_table = source_table.dropna(axis=1, how="all")

    standardized = source_table.rename(columns=SOURCE_TO_STANDARDIZED_COLUMN)
    standardized.insert(0, "source_year", int(config["source_year"]))
    standardized.insert(1, "source_file", str(config["source_file"]))
    standardized.insert(2, "source_sheet", str(config["source_sheet"]))
    standardized.insert(3, "source_row", standardized.pop("_source_row").astype(int))

    duplicate_columns_after_rename = standardized.columns[standardized.columns.duplicated()].tolist()
    if duplicate_columns_after_rename:
        raise ValueError(
            "Source-to-target renaming created duplicate standardized column(s): "
            f"{duplicate_columns_after_rename}"
        )

    for curated_field in LOCKED_CURATED_SCHEMA:
        if curated_field not in standardized.columns:
            standardized[curated_field] = pd.NA

    standardized = standardized[LOCKED_CURATED_SCHEMA].copy()

    # Stabilize sparse later-year integer columns before cross-year concatenation.
    # Older years intentionally contain structural nulls in these columns; explicit
    # nullable integer dtypes keep the concat result deterministic across pandas
    # dtype-inference rules and retain missing values without coercing to float.
    for column_name in CONCAT_STABLE_NULLABLE_INTEGER_FIELDS:
        standardized[column_name] = pd.to_numeric(
            standardized[column_name],
            errors="raise",
        ).astype("Int64")

    return standardized.reset_index(drop=True)

,source_year,source_file,source_sheet,excel_header_row,pandas_header_index
0,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,1,0
1,2021,resultat-ansokningsomgang-2021.xlsx,Tabell 3,1,0
2,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,1,0
3,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,6,5
4,2024,resultat-ansokningsomgang-2024.xlsx,Tabell 3,6,5
5,2025,resultat-ansokningsomgang-2025.xlsx,Tabell 3,7,6


### 8.2 Standardize every year and concatenate the preliminary applications base

The function is now applied once per configured year. Each yearly result is aligned to the locked 32-field schema before concatenation. Fields that the source workbook does not contain are intentionally created as structural-null placeholders. The normalization and convenience-derived fields remain schema placeholders on the ingestion-stage table; Section 9.2 populates them downstream in `curated_applications` without mutating that ingestion evidence.


In [16]:
standardized_tabell_3_by_year = {
    int(config["source_year"]): read_standardized_tabell_3(config)
    for config in source_import_config.sort_values("source_year").to_dict("records")
}

standardized_applications = pd.concat(
    [standardized_tabell_3_by_year[year] for year in EXPECTED_SOURCE_YEARS],
    ignore_index=True,
)

standardized_row_count_summary = pd.DataFrame(
    [
        {
            "source_year": year,
            "standardized_rows": len(table),
            "standardized_columns": len(table.columns),
        }
        for year, table in standardized_tabell_3_by_year.items()
    ]
).sort_values("source_year").reset_index(drop=True)

display(standardized_row_count_summary)
print(
    "Combined preliminary standardized applications table: "
    f"{standardized_applications.shape[0]:,} rows × {standardized_applications.shape[1]} columns"
)
display(standardized_applications.head())

,source_year,standardized_rows,standardized_columns
0,2020,1482,32
1,2021,1238,32
2,2022,1207,32
3,2023,1258,32
4,2024,1272,32
5,2025,1184,32


Combined preliminary standardized applications table: 7,641 rows × 32 columns


,source_year,source_file,source_sheet,source_row,diarienummer,utbildningsnamn,utbildningsomrade,beslut,beslut_normalized,is_approved,lan,kommun,flera_kommuner,has_multiple_municipalities,antal_kommuner,yh_poang,studieform,is_distance_based,studietakt_procent,examenstyp,utbildningsanordnare,huvudmannatyp,huvudmannatyp_normalized,sokta_utbildningsomgangar,beviljade_utbildningsomgangar,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade,sokta_platser_per_utbildningsomgang,sokta_platser_totalt,beviljade_platser_totalt
0,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,2,MYH 2020/4419,.NET Developer,Data/IT,Ej beviljad,<NA>,<NA>,Flera kommuner,Flera kommuner,Ja,<NA>,5,425,Bunden,<NA>,100,<NA>,KYH AB,Privat,<NA>,5,0,NaN,NaN,<NA>,NaN,<NA>,<NA>,<NA>
1,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,3,MYH 2020/4482,.NET Developer,Data/IT,Ej beviljad,<NA>,<NA>,Skåne,Malmö,Nej,<NA>,1,430,Bunden,<NA>,100,<NA>,KYH AB Malmö,Privat,<NA>,3,0,NaN,NaN,<NA>,NaN,<NA>,<NA>,<NA>
2,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,4,MYH 2020/5610,.net utvecklare,Data/IT,Ej beviljad,<NA>,<NA>,Västra Götaland,Göteborg,Nej,<NA>,1,400,Bunden,<NA>,100,<NA>,ABF Göteborg Vuxenutbildning AB,Privat,<NA>,3,0,NaN,NaN,<NA>,NaN,<NA>,<NA>,<NA>
3,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,5,MYH 2020/4403,.NET Utvecklare,Data/IT,Beviljad,<NA>,<NA>,Västra Götaland,Göteborg,Nej,<NA>,1,400,Bunden,<NA>,100,<NA>,Plushögskolan AB - Teknikhögskolan,Privat,<NA>,5,3,NaN,NaN,<NA>,NaN,<NA>,<NA>,<NA>
4,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,6,MYH 2020/5766,.NET-utvecklare,Data/IT,Beviljad,<NA>,<NA>,Stockholm,Stockholm,Nej,<NA>,1,400,Distans,<NA>,100,<NA>,IT-Högskolan Stockholm AB,Privat,<NA>,3,3,NaN,NaN,<NA>,NaN,<NA>,<NA>,<NA>


### 8.3 Ingestion integrity checks

These checks are intentionally ingestion-scoped rather than the broader curated-table validation handled later in Section 10. They confirm that the standardized import layer obeys the schema contract before the later cleaning/enrichment work begins.


In [17]:
expected_row_counts = tabell_3_profile[["source_year", "tabell_3_rows"]].rename(
    columns={"tabell_3_rows": "expected_rows"}
)
observed_row_counts = standardized_applications.groupby("source_year").size().rename("observed_rows").reset_index()
row_count_validation = expected_row_counts.merge(observed_row_counts, on="source_year", how="outer")
row_count_validation["row_count_matches"] = (
    row_count_validation["expected_rows"] == row_count_validation["observed_rows"]
)

combined_schema_order_ok = list(standardized_applications.columns) == LOCKED_CURATED_SCHEMA
per_year_schema_order_ok = all(
    list(table.columns) == LOCKED_CURATED_SCHEMA
    for table in standardized_tabell_3_by_year.values()
)
portable_source_file_ok = standardized_applications["source_file"].map(
    lambda value: str(value) == Path(str(value)).name and "/" not in str(value) and "\\" not in str(value)
).all()
source_sheet_ok = standardized_applications["source_sheet"].eq("Tabell 3").all()
source_year_scope_ok = sorted(standardized_applications["source_year"].unique().tolist()) == EXPECTED_SOURCE_YEARS
source_row_present_ok = standardized_applications["source_row"].notna().all()
missing_diarienummer_count = int(standardized_applications["diarienummer"].isna().sum())
duplicate_application_key_count = int(
    standardized_applications.duplicated(["source_year", "diarienummer"]).sum()
)

later_year_only_fields = [
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "seqf_niva",
    "smalt_yrkesomrade",
    "sokta_platser_per_utbildningsomgang",
    "sokta_platser_totalt",
    "beviljade_platser_totalt",
]

examenstyp_structural_null_ok = standardized_applications.loc[
    standardized_applications["source_year"] == 2020,
    "examenstyp",
].isna().all()

later_year_structural_null_ok = standardized_applications.loc[
    standardized_applications["source_year"].isin([2020, 2021, 2022]),
    later_year_only_fields,
].isna().all().all()

ingestion_placeholder_fields_unpopulated_ok = standardized_applications[
    INGESTION_STAGE_PLACEHOLDER_FIELDS
].isna().all().all()

ingestion_validation_summary = pd.DataFrame(
    [
        {"check": "Locked 32-column schema order on combined table", "passed": combined_schema_order_ok},
        {"check": "Locked schema order on every yearly table", "passed": per_year_schema_order_ok},
        {"check": "Configured source years match 2020–2025", "passed": source_year_scope_ok},
        {"check": "Portable source_file values use filenames only", "passed": portable_source_file_ok},
        {"check": "source_sheet is Tabell 3 for every row", "passed": source_sheet_ok},
        {"check": "source_row populated for every standardized row", "passed": source_row_present_ok},
        {"check": "No missing Diarienummer", "passed": missing_diarienummer_count == 0},
        {"check": "No duplicate (source_year, diarienummer) keys", "passed": duplicate_application_key_count == 0},
        {"check": "2020 examenstyp structural nulls preserved", "passed": examenstyp_structural_null_ok},
        {"check": "2020–2022 later-year-only structural nulls preserved", "passed": later_year_structural_null_ok},
        {"check": "Ingestion-stage normalization and convenience fields remain unpopulated", "passed": ingestion_placeholder_fields_unpopulated_ok},
    ]
)

display(row_count_validation)
display(ingestion_validation_summary)

assert row_count_validation["row_count_matches"].all(), (
    "Each standardized yearly table must preserve the profiled Tabell 3 row count."
)
assert combined_schema_order_ok and per_year_schema_order_ok, (
    "Standardized ingestion must enforce the locked 32-field schema order."
)
assert source_year_scope_ok, "The standardized applications table must cover exactly the expected 2020–2025 years."
assert portable_source_file_ok, "source_file should contain portable filenames only, not absolute paths."
assert source_sheet_ok, "The standardized ingestion table should only contain Tabell 3 rows."
assert source_row_present_ok, "source_row must be populated for every standardized row."
assert missing_diarienummer_count == 0, "Unexpected missing Diarienummer after standardized ingestion."
assert duplicate_application_key_count == 0, "Unexpected duplicate application keys after standardized ingestion."
assert examenstyp_structural_null_ok, "2020 should retain structural nulls for examenstyp."
assert later_year_structural_null_ok, "2020–2022 should retain structural nulls for later-year-only fields."
assert ingestion_placeholder_fields_unpopulated_ok, (
    "Normalization and selected convenience fields must remain unpopulated in standardized_applications."
)

,source_year,expected_rows,observed_rows,row_count_matches
0,2020,1482,1482,True
1,2021,1238,1238,True
2,2022,1207,1207,True
3,2023,1258,1258,True
4,2024,1272,1272,True
5,2025,1184,1184,True


,check,passed
0,Locked 32-column schema order on combined table,True
1,Locked schema order on every yearly table,True
2,Configured source years match 2020–2025,True
3,Portable source_file values use filenames only,True
4,source_sheet is Tabell 3 for every row,True
5,source_row populated for every standardized row,True
6,No missing Diarienummer,True
7,"No duplicate (source_year, diarienummer) keys",True
8,2020 examenstyp structural nulls preserved,True
9,2020–2022 later-year-only structural nulls pre...,True


## 9. Cleaning, normalization, and enrichment *(Sub-project 2.5)*

The standardized ingestion table is intentionally close to the source. This section performs the next layer of the workflow:
- convert imported representations into more analysis-ready datatypes,
- remove only whitespace artifacts that do not carry source meaning,
- populate the normalization fields and derived boolean fields already specified in Sub-project 2.3.

The separation matters. The ingestion layer proves that all six raw Excel workbooks can be read and aligned without interpretation. The cleaning/enrichment layer then makes that aligned table more consistent for longitudinal comparison, later validation, and later SQL/API use.

### 9.1 Datatype conversion and safe text cleanup

The combined `standardized_applications` table already has stable field names because the Sub-project 2.4 ingestion reader pre-stabilizes its four structurally sparse later-year integer fields as nullable `Int64` before cross-year concatenation. This 2.5 cleaning layer keeps that inherited dtype contract explicit and improves source-text comparison safety:
- `seqf_niva`, `sokta_platser_per_utbildningsomgang`, `sokta_platser_totalt`, and `beviljade_platser_totalt` must retain nullable `Int64` dtypes so structural nulls remain valid,
- several text fields contain harmless but comparison-breaking whitespace noise, such as trailing spaces in names.

The cleanup here is deliberately conservative:
- trim leading/trailing whitespace,
- collapse repeated whitespace to a single space,
- preserve missing values as missing,
- avoid case changes, accent removal, or semantic rewriting.

This improves equality checks and downstream grouping without changing the substantive source text. The text helper also moves cleaned text fields into pandas' nullable `string` dtype, while the numeric step revalidates the sparse `Int64` contract rather than relying on version-sensitive concat inference.


In [18]:
TEXT_COLUMNS_TO_CLEAN = [
    "source_file",
    "source_sheet",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "lan",
    "kommun",
    "flera_kommuner",
    "studieform",
    "examenstyp",
    "utbildningsanordnare",
    "huvudmannatyp",
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "smalt_yrkesomrade",
]

NULLABLE_INTEGER_COLUMNS = [
    "seqf_niva",
    "sokta_platser_per_utbildningsomgang",
    "sokta_platser_totalt",
    "beviljade_platser_totalt",
]


def clean_text_values(series: pd.Series) -> pd.Series:
    """Return conservatively cleaned nullable text without changing substantive wording."""
    cleaned = series.astype("string")
    cleaned = cleaned.str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned.mask(cleaned.eq(""), pd.NA)


def convert_to_nullable_integer(series: pd.Series, column_name: str) -> pd.Series:
    """Convert integer-like imported values to nullable Int64 while preserving structural nulls."""
    numeric = pd.to_numeric(series, errors="raise")
    non_null_values = numeric.dropna()
    non_integer_values = non_null_values[~np.isclose(non_null_values % 1, 0)]
    if not non_integer_values.empty:
        raise ValueError(
            f"{column_name} contains non-integer numeric value(s): "
            f"{sorted(non_integer_values.unique().tolist())[:5]}"
        )
    return numeric.astype("Int64")


def apply_cleaning_layer(standardized_input: pd.DataFrame) -> pd.DataFrame:
    """Create the cleaned 2.5 working table without mutating standardized ingestion output."""
    cleaned = standardized_input.copy(deep=True)

    for column_name in TEXT_COLUMNS_TO_CLEAN:
        cleaned[column_name] = clean_text_values(cleaned[column_name])

    for column_name in NULLABLE_INTEGER_COLUMNS:
        cleaned[column_name] = convert_to_nullable_integer(cleaned[column_name], column_name)

    return cleaned


cleaned_applications = apply_cleaning_layer(standardized_applications)

datatype_conversion_summary = pd.DataFrame(
    [
        {
            "column": column_name,
            "dtype_before_cleaning": str(standardized_applications[column_name].dtype),
            "dtype_after_cleaning": str(cleaned_applications[column_name].dtype),
            "non_null_values_after_cleaning": int(cleaned_applications[column_name].notna().sum()),
        }
        for column_name in NULLABLE_INTEGER_COLUMNS
    ]
)

text_cleanup_summary = pd.DataFrame(
    [
        {
            "column": column_name,
            "changed_values": int(
                (
                    standardized_applications[column_name].astype("string").fillna("<NA>")
                    != cleaned_applications[column_name].astype("string").fillna("<NA>")
                ).sum()
            ),
            "dtype_after_cleaning": str(cleaned_applications[column_name].dtype),
        }
        for column_name in TEXT_COLUMNS_TO_CLEAN
    ]
).sort_values(["changed_values", "column"], ascending=[False, True]).reset_index(drop=True)

print(
    "Cleaned applications working table: "
    f"{cleaned_applications.shape[0]:,} rows × {cleaned_applications.shape[1]} columns"
)
display(datatype_conversion_summary)
display(text_cleanup_summary.loc[text_cleanup_summary["changed_values"] > 0])

Cleaned applications working table: 7,641 rows × 32 columns


,column,dtype_before_cleaning,dtype_after_cleaning,non_null_values_after_cleaning
0,seqf_niva,Int64,Int64,3661
1,sokta_platser_per_utbildningsomgang,Int64,Int64,3714
2,sokta_platser_totalt,Int64,Int64,3714
3,beviljade_platser_totalt,Int64,Int64,3714


,column,changed_values,dtype_after_cleaning
0,utbildningsnamn,565,string
1,utbildningsanordnare,109,string
2,sun5_inriktning_namn,6,string


### 9.2 Normalization and useful derived fields

The 2.3 design intentionally retained both source-facing categories and normalized analytical companions. That pattern is implemented here:
- `beslut` remains the original source decision wording, while `beslut_normalized` becomes a stable longitudinal category,
- `huvudmannatyp` remains the source wording, while `huvudmannatyp_normalized` harmonizes `Landsting` and `Region`,
- three boolean convenience fields are populated so common questions become direct filters rather than repeated notebook logic.

The mapping helper validates coverage before assigning values. If a future workbook introduces an unexpected non-null category, the notebook should fail clearly instead of silently creating partial mappings.

In [19]:
STUDIEFORM_TO_IS_DISTANCE_BASED = {
    "Distans": True,
    "Bunden": False,
}

FLERA_KOMMUNER_TO_BOOLEAN = {
    "Ja": True,
    "Nej": False,
}

BESLUT_NORMALIZED_TO_IS_APPROVED = {
    "approved": True,
    "rejected": False,
    "withdrawn": False,
}


def map_values_with_coverage(
    series: pd.Series,
    mapping: dict[str, object],
    field_name: str,
) -> pd.Series:
    """Map non-null source categories only when every observed value is explicitly covered."""
    observed_values = {str(value) for value in series.dropna().unique()}
    unmapped_values = sorted(observed_values - set(mapping))
    if unmapped_values:
        raise ValueError(f"{field_name} contains unmapped value(s): {unmapped_values}")
    return series.map(mapping)


def add_normalization_and_enrichment(cleaned_input: pd.DataFrame) -> pd.DataFrame:
    """Populate the normalized categories and 2.5 boolean convenience fields."""
    curated = cleaned_input.copy(deep=True)

    curated["beslut_normalized"] = map_values_with_coverage(
        curated["beslut"],
        BESLUT_NORMALIZATION,
        "beslut",
    ).astype("string")

    curated["huvudmannatyp_normalized"] = map_values_with_coverage(
        curated["huvudmannatyp"],
        HUVUDMANNATYP_NORMALIZATION,
        "huvudmannatyp",
    ).astype("string")

    curated["is_approved"] = map_values_with_coverage(
        curated["beslut_normalized"],
        BESLUT_NORMALIZED_TO_IS_APPROVED,
        "beslut_normalized",
    ).astype("boolean")

    curated["is_distance_based"] = map_values_with_coverage(
        curated["studieform"],
        STUDIEFORM_TO_IS_DISTANCE_BASED,
        "studieform",
    ).astype("boolean")

    curated["has_multiple_municipalities"] = map_values_with_coverage(
        curated["flera_kommuner"],
        FLERA_KOMMUNER_TO_BOOLEAN,
        "flera_kommuner",
    ).astype("boolean")

    return curated


curated_applications = add_normalization_and_enrichment(cleaned_applications)

beslut_normalization_result = (
    curated_applications.groupby(["beslut", "beslut_normalized"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["beslut_normalized", "beslut"])
    .reset_index(drop=True)
)

huvudmannatyp_normalization_result = (
    curated_applications.groupby(["huvudmannatyp", "huvudmannatyp_normalized"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["huvudmannatyp_normalized", "huvudmannatyp"])
    .reset_index(drop=True)
)

derived_boolean_summary = pd.DataFrame(
    [
        {
            "field": column_name,
            "true_rows": int(curated_applications[column_name].eq(True).sum()),
            "false_rows": int(curated_applications[column_name].eq(False).sum()),
            "missing_rows": int(curated_applications[column_name].isna().sum()),
            "dtype": str(curated_applications[column_name].dtype),
        }
        for column_name in [
            "is_approved",
            "is_distance_based",
            "has_multiple_municipalities",
        ]
    ]
)

print(
    "Curated 2.5 applications table: "
    f"{curated_applications.shape[0]:,} rows × {curated_applications.shape[1]} columns"
)
display(beslut_normalization_result)
display(huvudmannatyp_normalization_result)
display(derived_boolean_summary)

Curated 2.5 applications table: 7,641 rows × 32 columns


,beslut,beslut_normalized,rows
0,Beviljad,approved,2613
1,Avslag,rejected,3217
2,Ej beviljad,rejected,1810
3,Återkallad,withdrawn,1


,huvudmannatyp,huvudmannatyp_normalized,rows
0,Kommun,Kommun,1261
1,Privat,Privat,6284
2,Landsting,Region,23
3,Region,Region,60
4,Statlig,Statlig,13


,field,true_rows,false_rows,missing_rows,dtype
0,is_approved,2613,5028,0,boolean
1,is_distance_based,3562,4079,0,boolean
2,has_multiple_municipalities,1433,6208,0,boolean


### 9.3 Rerun safety and transformation integrity checks

This transformation layer must be safe to rerun from the same standardized input. Instead of mutating the ingestion-stage table, the notebook uses copy-based transformation functions and verifies that:
- repeated execution produces the same curated table,
- the 32-field schema and row count stay unchanged,
- the original `standardized_applications` table remains unpopulated in the fields that cleaning/enrichment is responsible for filling,
- conservative text cleanup actually removes outer/repeated whitespace,
- nullable integer and nullable boolean dtypes are present where intended,
- the normalized and derived fields are populated completely for the observed source categories.

These checks are transformation-scoped. The broader dataset validation and missingness review are handled later in Section 10; the export discussion belongs in Section 11.


In [20]:
curated_applications_rerun_check = add_normalization_and_enrichment(
    apply_cleaning_layer(standardized_applications)
)

transformation_rerun_safe_ok = curated_applications.equals(curated_applications_rerun_check)
row_count_preserved_after_cleaning_ok = len(curated_applications) == len(standardized_applications)
schema_order_preserved_after_cleaning_ok = list(curated_applications.columns) == LOCKED_CURATED_SCHEMA
standardized_input_preserved_ok = standardized_applications[
    INGESTION_STAGE_PLACEHOLDER_FIELDS
].isna().all().all()

standardized_sparse_integer_dtypes_ok = all(
    str(standardized_applications[column_name].dtype) == "Int64"
    for column_name in CONCAT_STABLE_NULLABLE_INTEGER_FIELDS
)

cleaned_text_whitespace_ok = all(
    not curated_applications[column_name]
    .dropna()
    .astype("string")
    .str.contains(r"^\s|\s$|\s{2,}", regex=True)
    .any()
    for column_name in TEXT_COLUMNS_TO_CLEAN
)

nullable_integer_dtypes_ok = all(
    str(curated_applications[column_name].dtype) == "Int64"
    for column_name in NULLABLE_INTEGER_COLUMNS
)

nullable_boolean_dtypes_ok = all(
    str(curated_applications[column_name].dtype) == "boolean"
    for column_name in [
        "is_approved",
        "is_distance_based",
        "has_multiple_municipalities",
    ]
)

normalized_and_derived_fields_populated_ok = curated_applications[
    [
        "beslut_normalized",
        "huvudmannatyp_normalized",
        "is_approved",
        "is_distance_based",
        "has_multiple_municipalities",
    ]
].notna().all().all()

normalized_value_domains_ok = (
    set(curated_applications["beslut_normalized"].dropna().unique())
    <= set(BESLUT_NORMALIZATION.values())
    and set(curated_applications["huvudmannatyp_normalized"].dropna().unique())
    <= set(HUVUDMANNATYP_NORMALIZATION.values())
)

transformation_validation_summary = pd.DataFrame(
    [
        {"check": "Repeated 2.5 transformation gives the same curated table", "passed": transformation_rerun_safe_ok},
        {"check": "2.5 transformation preserves the 7,641-row input count", "passed": row_count_preserved_after_cleaning_ok},
        {"check": "2.5 transformation preserves locked 32-field schema order", "passed": schema_order_preserved_after_cleaning_ok},
        {"check": "Standardized ingestion table remains unmodified in 2.5-derived fields", "passed": standardized_input_preserved_ok},
        {"check": "Standardized sparse later-year integer fields are concat-stable nullable Int64", "passed": standardized_sparse_integer_dtypes_ok},
        {"check": "Safe text cleanup removes outer and repeated whitespace", "passed": cleaned_text_whitespace_ok},
        {"check": "Selected later-year numeric fields use nullable Int64", "passed": nullable_integer_dtypes_ok},
        {"check": "Derived convenience flags use nullable boolean dtype", "passed": nullable_boolean_dtypes_ok},
        {"check": "Normalized and derived 2.5 fields are populated for observed categories", "passed": normalized_and_derived_fields_populated_ok},
        {"check": "Normalized value domains remain within the locked 2.3 mappings", "passed": normalized_value_domains_ok},
    ]
)

display(transformation_validation_summary)
display(curated_applications.head())

assert transformation_rerun_safe_ok, "Repeating the 2.5 transformation should reproduce the same curated table."
assert row_count_preserved_after_cleaning_ok, "Cleaning/enrichment should not add or remove application rows."
assert schema_order_preserved_after_cleaning_ok, "Cleaning/enrichment must preserve the locked 32-field schema order."
assert standardized_input_preserved_ok, "The 2.5 layer should not mutate standardized_applications in place."
assert standardized_sparse_integer_dtypes_ok, "Sparse later-year integer fields should be nullable Int64 before and after concatenation."
assert cleaned_text_whitespace_ok, "Text cleanup should remove outer and repeated whitespace from configured text columns."
assert nullable_integer_dtypes_ok, "Selected later-year numeric fields should use nullable Int64 after cleaning."
assert nullable_boolean_dtypes_ok, "Derived convenience flags should use pandas nullable boolean dtype."
assert normalized_and_derived_fields_populated_ok, "Observed 2.5 normalization and boolean outputs should be fully populated."
assert normalized_value_domains_ok, "Normalized values must stay inside the locked 2.3 mapping domains."

,check,passed
0,Repeated 2.5 transformation gives the same cur...,True
1,"2.5 transformation preserves the 7,641-row inp...",True
2,2.5 transformation preserves locked 32-field s...,True
3,Standardized ingestion table remains unmodifie...,True
4,Standardized sparse later-year integer fields ...,True
5,Safe text cleanup removes outer and repeated w...,True
6,Selected later-year numeric fields use nullabl...,True
7,Derived convenience flags use nullable boolean...,True
8,Normalized and derived 2.5 fields are populate...,True
9,Normalized value domains remain within the loc...,True


,source_year,source_file,source_sheet,source_row,diarienummer,utbildningsnamn,utbildningsomrade,beslut,beslut_normalized,is_approved,lan,kommun,flera_kommuner,has_multiple_municipalities,antal_kommuner,yh_poang,studieform,is_distance_based,studietakt_procent,examenstyp,utbildningsanordnare,huvudmannatyp,huvudmannatyp_normalized,sokta_utbildningsomgangar,beviljade_utbildningsomgangar,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade,sokta_platser_per_utbildningsomgang,sokta_platser_totalt,beviljade_platser_totalt
0,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,2,MYH 2020/4419,.NET Developer,Data/IT,Ej beviljad,rejected,False,Flera kommuner,Flera kommuner,Ja,True,5,425,Bunden,False,100,<NA>,KYH AB,Privat,Privat,5,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,3,MYH 2020/4482,.NET Developer,Data/IT,Ej beviljad,rejected,False,Skåne,Malmö,Nej,False,1,430,Bunden,False,100,<NA>,KYH AB Malmö,Privat,Privat,3,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,4,MYH 2020/5610,.net utvecklare,Data/IT,Ej beviljad,rejected,False,Västra Götaland,Göteborg,Nej,False,1,400,Bunden,False,100,<NA>,ABF Göteborg Vuxenutbildning AB,Privat,Privat,3,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,5,MYH 2020/4403,.NET Utvecklare,Data/IT,Beviljad,approved,True,Västra Götaland,Göteborg,Nej,False,1,400,Bunden,False,100,<NA>,Plushögskolan AB - Teknikhögskolan,Privat,Privat,5,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,6,MYH 2020/5766,.NET-utvecklare,Data/IT,Beviljad,approved,True,Stockholm,Stockholm,Nej,False,1,400,Distans,True,100,<NA>,IT-Högskolan Stockholm AB,Privat,Privat,3,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 10. Validation and quality checks *(Sub-project 2.6)*

The transformation layer proves that the workflow is deterministic. This validation layer asks a broader question: **is the curated dataset trustworthy enough to export and hand to later SQL/API work?**

The checks below therefore focus on dataset-level assurances that matter downstream:
- the curated table still has the intended application grain,
- source traceability remains intact,
- key identifiers are complete and unique at the chosen grain,
- normalized categories and derived convenience flags still obey their documented meanings,
- missingness is reviewed as a data-quality topic rather than hidden by a single total.

These checks are intentionally narrow and decision-linked. They validate the agreed curated table instead of reopening the earlier source-selection or normalization design.


### 10.1 Curated-table identity, traceability, and semantic checks

A curated export should not merely *exist*; it should preserve the modeling commitments made earlier in the notebook. The checks in this block verify that the final working table:
- stays inside the locked 32-field schema and agreed six-year scope,
- preserves the one-row-per-application grain through the `(source_year, diarienummer)` key,
- preserves portable source-row traceability,
- contains no missing values in the fields needed to identify and interpret a row,
- keeps normalization and boolean helper fields aligned with their mapping rules.

This creates an explicit quality gate before any processed file is written.

In [21]:
CURATED_REQUIRED_NON_NULL_FIELDS = [
    "source_year",
    "source_file",
    "source_sheet",
    "source_row",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "beslut_normalized",
    "is_approved",
    "utbildningsanordnare",
    "huvudmannatyp",
    "huvudmannatyp_normalized",
]

SOURCE_TRACEABILITY_KEY = ["source_year", "source_file", "source_sheet", "source_row"]
APPLICATION_GRAIN_KEY = ["source_year", "diarienummer"]

curated_regenerated_for_validation = add_normalization_and_enrichment(
    apply_cleaning_layer(standardized_applications)
)

curated_rerun_safe_ok = curated_applications.equals(curated_regenerated_for_validation)
curated_schema_order_ok = list(curated_applications.columns) == LOCKED_CURATED_SCHEMA
curated_row_count_ok = len(curated_applications) == int(row_count_validation["expected_rows"].sum())
curated_source_year_scope_ok = sorted(curated_applications["source_year"].unique().tolist()) == EXPECTED_SOURCE_YEARS
curated_source_sheet_contract_ok = curated_applications["source_sheet"].eq("Tabell 3").all()
curated_required_fields_complete_ok = curated_applications[CURATED_REQUIRED_NON_NULL_FIELDS].notna().all().all()

missing_required_value_count = int(
    curated_applications[CURATED_REQUIRED_NON_NULL_FIELDS].isna().sum().sum()
)
missing_diarienummer_count_curated = int(curated_applications["diarienummer"].isna().sum())
duplicate_application_key_count_curated = int(
    curated_applications.duplicated(APPLICATION_GRAIN_KEY).sum()
)
duplicate_source_identity_count_curated = int(
    curated_applications.duplicated(SOURCE_TRACEABILITY_KEY).sum()
)

curated_application_key_unique_ok = duplicate_application_key_count_curated == 0
curated_source_identity_unique_ok = duplicate_source_identity_count_curated == 0
curated_diarienummer_complete_ok = missing_diarienummer_count_curated == 0

beslut_domain_ok = (
    set(curated_applications["beslut_normalized"].dropna().unique())
    <= set(BESLUT_NORMALIZATION.values())
)
huvudmannatyp_domain_ok = (
    set(curated_applications["huvudmannatyp_normalized"].dropna().unique())
    <= set(HUVUDMANNATYP_NORMALIZATION.values())
)

is_approved_consistency_ok = curated_applications["is_approved"].equals(
    curated_applications["beslut_normalized"]
    .map(BESLUT_NORMALIZED_TO_IS_APPROVED)
    .astype("boolean")
)
is_distance_based_consistency_ok = curated_applications["is_distance_based"].equals(
    curated_applications["studieform"]
    .map(STUDIEFORM_TO_IS_DISTANCE_BASED)
    .astype("boolean")
)
has_multiple_municipalities_consistency_ok = curated_applications["has_multiple_municipalities"].equals(
    curated_applications["flera_kommuner"]
    .map(FLERA_KOMMUNER_TO_BOOLEAN)
    .astype("boolean")
)

curated_validation_summary = pd.DataFrame(
    [
        {
            "check": "Curated table can be deterministically rebuilt from the standardized base",
            "passed": curated_rerun_safe_ok,
            "evidence": "Direct equality against a fresh cleaning + enrichment rerun",
        },
        {
            "check": "Locked 32-field curated schema order is preserved",
            "passed": curated_schema_order_ok,
            "evidence": f"{curated_applications.shape[1]} columns in locked order",
        },
        {
            "check": "Curated row count still matches the profiled Tabell 3 total",
            "passed": curated_row_count_ok,
            "evidence": f"{len(curated_applications):,} rows",
        },
        {
            "check": "Curated year scope remains exactly 2020–2025",
            "passed": curated_source_year_scope_ok,
            "evidence": ", ".join(map(str, EXPECTED_SOURCE_YEARS)),
        },
        {
            "check": "Every curated row still points to Tabell 3",
            "passed": curated_source_sheet_contract_ok,
            "evidence": "source_sheet == 'Tabell 3' for every row",
        },
        {
            "check": "Required identification and interpretation fields are complete",
            "passed": curated_required_fields_complete_ok,
            "evidence": f"{missing_required_value_count} missing required values",
        },
        {
            "check": "Diarienummer is complete on the curated table",
            "passed": curated_diarienummer_complete_ok,
            "evidence": f"{missing_diarienummer_count_curated} missing diarienummer values",
        },
        {
            "check": "Application grain key (source_year, diarienummer) is unique",
            "passed": curated_application_key_unique_ok,
            "evidence": f"{duplicate_application_key_count_curated} duplicate key rows",
        },
        {
            "check": "Source-row traceability identity is unique",
            "passed": curated_source_identity_unique_ok,
            "evidence": f"{duplicate_source_identity_count_curated} duplicate source-row identities",
        },
        {
            "check": "Normalized decision domain stays inside the locked mapping",
            "passed": beslut_domain_ok,
            "evidence": ", ".join(sorted(curated_applications["beslut_normalized"].dropna().unique())),
        },
        {
            "check": "Normalized provider-type domain stays inside the locked mapping",
            "passed": huvudmannatyp_domain_ok,
            "evidence": ", ".join(sorted(curated_applications["huvudmannatyp_normalized"].dropna().unique())),
        },
        {
            "check": "is_approved matches beslut_normalized semantics",
            "passed": is_approved_consistency_ok,
            "evidence": "approved -> True; rejected/withdrawn -> False",
        },
        {
            "check": "is_distance_based matches studieform semantics",
            "passed": is_distance_based_consistency_ok,
            "evidence": "Distans -> True; Bunden -> False",
        },
        {
            "check": "has_multiple_municipalities matches flera_kommuner semantics",
            "passed": has_multiple_municipalities_consistency_ok,
            "evidence": "Ja -> True; Nej -> False",
        },
    ]
)

display(curated_validation_summary)

assert curated_rerun_safe_ok, "The curated table should be reproducible from the standardized base."
assert curated_schema_order_ok, "The final curated schema order must stay locked."
assert curated_row_count_ok, "The curated row count should match the profiled Tabell 3 total."
assert curated_source_year_scope_ok, "The curated table must preserve the agreed 2020–2025 scope."
assert curated_source_sheet_contract_ok, "Every curated row should still trace back to Tabell 3."
assert curated_required_fields_complete_ok, "Required curated identity/interpretation fields must not be missing."
assert curated_diarienummer_complete_ok, "Diarienummer must be present on every curated row."
assert curated_application_key_unique_ok, "The curated application-grain key should be unique."
assert curated_source_identity_unique_ok, "Source-row traceability identities should be unique."
assert beslut_domain_ok, "Normalized decisions must remain inside the locked mapping domain."
assert huvudmannatyp_domain_ok, "Normalized provider types must remain inside the locked mapping domain."
assert is_approved_consistency_ok, "is_approved must agree with beslut_normalized."
assert is_distance_based_consistency_ok, "is_distance_based must agree with studieform."
assert has_multiple_municipalities_consistency_ok, "has_multiple_municipalities must agree with flera_kommuner."

,check,passed,evidence
0,Curated table can be deterministically rebuilt...,True,Direct equality against a fresh cleaning + enr...
1,Locked 32-field curated schema order is preserved,True,32 columns in locked order
2,Curated row count still matches the profiled T...,True,"7,641 rows"
3,Curated year scope remains exactly 2020–2025,True,"2020, 2021, 2022, 2023, 2024, 2025"
4,Every curated row still points to Tabell 3,True,source_sheet == 'Tabell 3' for every row
5,Required identification and interpretation fie...,True,0 missing required values
6,Diarienummer is complete on the curated table,True,0 missing diarienummer values
7,"Application grain key (source_year, diarienumm...",True,0 duplicate key rows
8,Source-row traceability identity is unique,True,0 duplicate source-row identities
9,Normalized decision domain stays inside the lo...,True,"approved, rejected, withdrawn"


### 10.2 Missingness review: structural nulls vs ordinary gaps

A missing-value total is not enough here, because several retained fields were intentionally absent in older source years. The missingness review therefore separates:
- **structural nulls** created by the agreed 2.3 schema policy for years where a field did not exist,
- **later-year source gaps** that remain visible rather than being silently imputed.

This matters for both analysis and SQL loading. A null in `sun5_inriktning` for 2021 means “that field was not available in that source year”; a null in `seqf_niva` for a later year means “the field exists, but this row has no recorded value.” Those are different data-quality facts.

In [22]:
STRUCTURAL_NULL_EXPECTATIONS = {
    "examenstyp": {
        "years": [2020],
        "review_note": "Exam type is not present in the 2020 Tabell 3 structure.",
    },
    "sun5_inriktning": {
        "years": [2020, 2021, 2022],
        "review_note": "SUN5 fields are retained later-year fields and are structurally null in 2020–2022.",
    },
    "sun5_inriktning_namn": {
        "years": [2020, 2021, 2022],
        "review_note": "SUN5 fields are retained later-year fields and are structurally null in 2020–2022.",
    },
    "seqf_niva": {
        "years": [2020, 2021, 2022],
        "review_note": "SeQF is structurally null in 2020–2022 and may still contain later-year source gaps.",
    },
    "smalt_yrkesomrade": {
        "years": [2020, 2021, 2022],
        "review_note": "The narrow occupational-area field is structurally null in 2020–2022.",
    },
    "sokta_platser_per_utbildningsomgang": {
        "years": [2020, 2021, 2022],
        "review_note": "Later-year seat-total summary field; structurally null in 2020–2022.",
    },
    "sokta_platser_totalt": {
        "years": [2020, 2021, 2022],
        "review_note": "Later-year seat-total summary field; structurally null in 2020–2022.",
    },
    "beviljade_platser_totalt": {
        "years": [2020, 2021, 2022],
        "review_note": "Later-year seat-total summary field; structurally null in 2020–2022.",
    },
}

missingness_review_rows = []
structural_null_pattern_checks = []
unexpected_missing_outside_declared_structural_fields = []

for column_name, spec in STRUCTURAL_NULL_EXPECTATIONS.items():
    structural_year_mask = curated_applications["source_year"].isin(spec["years"])
    outside_structural_year_mask = ~structural_year_mask

    total_missing = int(curated_applications[column_name].isna().sum())
    rows_in_structural_years = int(structural_year_mask.sum())
    missing_inside_structural_years = int(
        curated_applications.loc[structural_year_mask, column_name].isna().sum()
    )
    missing_outside_structural_years = int(
        curated_applications.loc[outside_structural_year_mask, column_name].isna().sum()
    )
    structural_block_complete = missing_inside_structural_years == rows_in_structural_years

    structural_null_pattern_checks.append(structural_block_complete)
    if missing_outside_structural_years > 0:
        unexpected_missing_outside_declared_structural_fields.append(column_name)

    if missing_outside_structural_years == 0:
        missingness_classification = "structural nulls only"
    else:
        missingness_classification = "structural nulls + later-year source gaps"

    missingness_review_rows.append(
        {
            "column": column_name,
            "total_missing": total_missing,
            "declared_structural_years": ", ".join(map(str, spec["years"])),
            "structural_missing_rows": missing_inside_structural_years,
            "missing_outside_structural_years": missing_outside_structural_years,
            "classification": missingness_classification,
            "structural_block_complete": structural_block_complete,
            "review_note": spec["review_note"],
        }
    )

missingness_review = pd.DataFrame(missingness_review_rows)

columns_with_missing_values = {
    column_name
    for column_name, missing_count in curated_applications.isna().sum().items()
    if int(missing_count) > 0
}
structural_null_columns = set(STRUCTURAL_NULL_EXPECTATIONS)
missing_columns_explained_by_review_ok = columns_with_missing_values <= structural_null_columns
structural_null_blocks_complete_ok = all(structural_null_pattern_checks)
non_structural_gap_scope_ok = set(unexpected_missing_outside_declared_structural_fields) <= {"seqf_niva"}
exam_type_only_structural_missing_ok = int(
    missingness_review.loc[
        missingness_review["column"].eq("examenstyp"),
        "missing_outside_structural_years",
    ].iloc[0]
) == 0
later_year_summary_fields_complete_when_present_ok = all(
    int(
        missingness_review.loc[
            missingness_review["column"].eq(column_name),
            "missing_outside_structural_years",
        ].iloc[0]
    ) == 0
    for column_name in [
        "sun5_inriktning",
        "sun5_inriktning_namn",
        "smalt_yrkesomrade",
        "sokta_platser_per_utbildningsomgang",
        "sokta_platser_totalt",
        "beviljade_platser_totalt",
    ]
)

missingness_quality_summary = pd.DataFrame(
    [
        {
            "check": "Every column with missing values is covered by the explicit structural-null review",
            "passed": missing_columns_explained_by_review_ok,
            "evidence": ", ".join(sorted(columns_with_missing_values)),
        },
        {
            "check": "Declared structural-null year blocks are completely null",
            "passed": structural_null_blocks_complete_ok,
            "evidence": "Older-year structural-null contracts match the source-design decisions",
        },
        {
            "check": "Missing values outside structural years are limited to the reviewed SeQF field",
            "passed": non_structural_gap_scope_ok,
            "evidence": ", ".join(sorted(unexpected_missing_outside_declared_structural_fields)) or "none",
        },
        {
            "check": "examenstyp has no missing values outside its declared 2020 structural-null scope",
            "passed": exam_type_only_structural_missing_ok,
            "evidence": "2021–2025 values are complete",
        },
        {
            "check": "Later-year retained summary/classification fields are complete once their source years begin",
            "passed": later_year_summary_fields_complete_when_present_ok,
            "evidence": "All reviewed later-year fields except seqf_niva have no later-year gaps",
        },
    ]
)

display(missingness_review)
display(missingness_quality_summary)

assert missing_columns_explained_by_review_ok, "Columns with missing values should be explicitly covered by the 2.6 review."
assert structural_null_blocks_complete_ok, "Declared structural-null year blocks should be fully null."
assert non_structural_gap_scope_ok, "Only the reviewed seqf_niva field should have later-year source gaps."
assert exam_type_only_structural_missing_ok, "examenstyp should be complete outside 2020."
assert later_year_summary_fields_complete_when_present_ok, "Retained later-year fields should be complete after their structural-null years, except reviewed seqf_niva gaps."

,column,total_missing,declared_structural_years,structural_missing_rows,missing_outside_structural_years,classification,structural_block_complete,review_note
0,examenstyp,1482,2020,1482,0,structural nulls only,True,Exam type is not present in the 2020 Tabell 3 ...
1,sun5_inriktning,3927,"2020, 2021, 2022",3927,0,structural nulls only,True,SUN5 fields are retained later-year fields and...
2,sun5_inriktning_namn,3927,"2020, 2021, 2022",3927,0,structural nulls only,True,SUN5 fields are retained later-year fields and...
3,seqf_niva,3980,"2020, 2021, 2022",3927,53,structural nulls + later-year source gaps,True,SeQF is structurally null in 2020–2022 and may...
4,smalt_yrkesomrade,3927,"2020, 2021, 2022",3927,0,structural nulls only,True,The narrow occupational-area field is structur...
5,sokta_platser_per_utbildningsomgang,3927,"2020, 2021, 2022",3927,0,structural nulls only,True,Later-year seat-total summary field; structura...
6,sokta_platser_totalt,3927,"2020, 2021, 2022",3927,0,structural nulls only,True,Later-year seat-total summary field; structura...
7,beviljade_platser_totalt,3927,"2020, 2021, 2022",3927,0,structural nulls only,True,Later-year seat-total summary field; structura...


,check,passed,evidence
0,Every column with missing values is covered by...,True,"beviljade_platser_totalt, examenstyp, seqf_niv..."
1,Declared structural-null year blocks are compl...,True,Older-year structural-null contracts match the...
2,Missing values outside structural years are li...,True,seqf_niva
3,examenstyp has no missing values outside its d...,True,2021–2025 values are complete
4,Later-year retained summary/classification fie...,True,All reviewed later-year fields except seqf_niv...


### 10.3 Year-wise sanity summaries and validation checkpoint

Longitudinal data should be checked year by year, not only in a single all-years total. The summary below verifies that:
- each year still contributes the expected number of rows,
- the application key stays unique inside each year,
- the normalized decision counts reconcile back to the yearly row count.

The final quality-gate table gathers the strongest validation checks into a compact export-readiness checkpoint.


In [23]:
decision_counts_by_year = (
    curated_applications.groupby(["source_year", "beslut_normalized"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["approved", "rejected", "withdrawn"], fill_value=0)
    .reset_index()
)

yearwise_curated_summary = (
    curated_applications.groupby("source_year")
    .agg(
        curated_rows=("diarienummer", "size"),
        unique_diarienummer=("diarienummer", "nunique"),
        approved_rows=("is_approved", lambda values: int(values.eq(True).sum())),
    )
    .reset_index()
    .merge(
        row_count_validation[["source_year", "expected_rows"]],
        on="source_year",
        how="left",
    )
    .merge(decision_counts_by_year, on="source_year", how="left")
    .sort_values("source_year")
    .reset_index(drop=True)
)

yearwise_curated_summary["row_count_matches_profile"] = (
    yearwise_curated_summary["curated_rows"] == yearwise_curated_summary["expected_rows"]
)
yearwise_curated_summary["key_count_matches_rows"] = (
    yearwise_curated_summary["curated_rows"] == yearwise_curated_summary["unique_diarienummer"]
)
yearwise_curated_summary["decision_counts_reconcile"] = (
    yearwise_curated_summary[["approved", "rejected", "withdrawn"]].sum(axis=1)
    == yearwise_curated_summary["curated_rows"]
)
yearwise_curated_summary["approval_rate_percent"] = (
    yearwise_curated_summary["approved_rows"]
    / yearwise_curated_summary["curated_rows"]
    * 100
).round(1)

yearwise_row_counts_match_profile_ok = yearwise_curated_summary["row_count_matches_profile"].all()
yearwise_key_counts_match_rows_ok = yearwise_curated_summary["key_count_matches_rows"].all()
yearwise_decision_counts_reconcile_ok = yearwise_curated_summary["decision_counts_reconcile"].all()

quality_gate_summary = pd.DataFrame(
    [
        {
            "quality_gate": "Core curated identity/domain checks",
            "passed": bool(curated_validation_summary["passed"].all()),
            "evidence": f"{int(curated_validation_summary['passed'].sum())}/{len(curated_validation_summary)} checks passed",
        },
        {
            "quality_gate": "Missingness review and structural-null interpretation",
            "passed": bool(missingness_quality_summary["passed"].all()),
            "evidence": f"{int(missingness_quality_summary['passed'].sum())}/{len(missingness_quality_summary)} checks passed",
        },
        {
            "quality_gate": "Year-wise row counts match the profiled Tabell 3 totals",
            "passed": bool(yearwise_row_counts_match_profile_ok),
            "evidence": "All six years reconcile",
        },
        {
            "quality_gate": "Year-wise application key counts match row counts",
            "passed": bool(yearwise_key_counts_match_rows_ok),
            "evidence": "No within-year duplicate application identifiers",
        },
        {
            "quality_gate": "Year-wise normalized decision counts reconcile to row counts",
            "passed": bool(yearwise_decision_counts_reconcile_ok),
            "evidence": "Approved + rejected + withdrawn equals yearly total",
        },
    ]
)

validated_curated_dataset_ready_ok = bool(quality_gate_summary["passed"].all())

display(yearwise_curated_summary)
display(quality_gate_summary)

assert yearwise_row_counts_match_profile_ok, "Each curated year should match the profiled source-row count."
assert yearwise_key_counts_match_rows_ok, "Each year's application identifiers should remain unique."
assert yearwise_decision_counts_reconcile_ok, "Decision counts should reconcile to each yearly row count."
assert validated_curated_dataset_ready_ok, "The curated dataset must pass the full 2.6 quality gate before export."

,source_year,curated_rows,unique_diarienummer,approved_rows,expected_rows,approved,rejected,withdrawn,row_count_matches_profile,key_count_matches_rows,decision_counts_reconcile,approval_rate_percent
0,2020,1482,1482,484,1482,484,998,0,True,True,True,32.7
1,2021,1238,1238,426,1238,426,812,0,True,True,True,34.4
2,2022,1207,1207,420,1207,420,787,0,True,True,True,34.8
3,2023,1258,1258,477,1258,477,781,0,True,True,True,37.9
4,2024,1272,1272,344,1272,344,928,0,True,True,True,27.0
5,2025,1184,1184,462,1184,462,721,1,True,True,True,39.0


,quality_gate,passed,evidence
0,Core curated identity/domain checks,True,14/14 checks passed
1,Missingness review and structural-null interpr...,True,5/5 checks passed
2,Year-wise row counts match the profiled Tabell...,True,All six years reconcile
3,Year-wise application key counts match row counts,True,No within-year duplicate application identifiers
4,Year-wise normalized decision counts reconcile...,True,Approved + rejected + withdrawn equals yearly ...


## 11. Export of the curated dataset *(Sub-project 2.6)*

The processed exports are written only after the validation quality gate has passed. This keeps every file in `data/processed/` tied to the same validated in-memory table rather than to an unchecked intermediate result.

The final export contract keeps **two complementary outputs** from the same `curated_applications` table:
- **CSV** for transparency, ordinary inspection, and simple downstream SQL loading,
- **Parquet** for a typed, machine-friendly companion artifact that is convenient for later programmatic reuse.

An earlier implementation draft treated CSV as the only canonical export and deferred Parquet. After rechecking the project requirement and the downstream handoff value, the final notebook retains **both CSV and Parquet** without changing the curated schema or table grain.


### 11.1 CSV and Parquet export contract

The paired export contract is intentionally explicit:
- **CSV file:** `part_2/data/processed/myh_curated_applications_2020_2025.csv`,
- **Parquet file:** `part_2/data/processed/myh_curated_applications_2020_2025.parquet`,
- **source DataFrame:** `curated_applications`, only after `validated_curated_dataset_ready_ok` is true,
- **grain:** one row = one application in one MYH application round,
- **scope:** 2020–2025,
- **schema:** the locked 32-field curated order,
- **index:** not exported in either file,
- **CSV specifics:** UTF-8 text, with missing values written as empty CSV fields,
- **Parquet specifics:** written with pandas `to_parquet(..., engine="pyarrow", compression="snappy", index=False)`.

The Parquet path deliberately names **PyArrow** as its engine. This keeps the output contract deterministic: if PyArrow is not installed in the local environment, the notebook raises a targeted installation message before refreshing processed files.


In [24]:
CURATED_CSV_FILE_NAME = "myh_curated_applications_2020_2025.csv"
CURATED_PARQUET_FILE_NAME = "myh_curated_applications_2020_2025.parquet"
CURATED_CSV_PATH = PROCESSED_DATA_DIR / CURATED_CSV_FILE_NAME
CURATED_PARQUET_PATH = PROCESSED_DATA_DIR / CURATED_PARQUET_FILE_NAME
CSV_EXPORT_ENCODING = "utf-8"
CSV_EXPORT_NULL_REPRESENTATION = ""
PARQUET_EXPORT_ENGINE = "pyarrow"
PARQUET_EXPORT_COMPRESSION = "snappy"

assert validated_curated_dataset_ready_ok, (
    "The final processed exports must only be written after the curated dataset passes the full 2.6 quality gate."
)

try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "Sub-project 2.6 now writes a Parquet companion export with engine='pyarrow'. "
        "Install PyArrow with `pip install pyarrow` and rerun the notebook."
    ) from exc

curated_applications.to_csv(
    CURATED_CSV_PATH,
    index=False,
    encoding=CSV_EXPORT_ENCODING,
    na_rep=CSV_EXPORT_NULL_REPRESENTATION,
)

curated_applications.to_parquet(
    CURATED_PARQUET_PATH,
    index=False,
    engine=PARQUET_EXPORT_ENGINE,
    compression=PARQUET_EXPORT_COMPRESSION,
)

csv_export_contract_summary = pd.DataFrame(
    [
        {"contract_item": "processed_file", "value": str(CURATED_CSV_PATH.relative_to(PART_2_DIR))},
        {"contract_item": "source_table", "value": "curated_applications"},
        {"contract_item": "row_count", "value": f"{len(curated_applications):,}"},
        {"contract_item": "column_count", "value": str(len(LOCKED_CURATED_SCHEMA))},
        {"contract_item": "schema_order", "value": "LOCKED_CURATED_SCHEMA"},
        {"contract_item": "csv_index", "value": "not exported"},
        {"contract_item": "encoding", "value": CSV_EXPORT_ENCODING},
        {"contract_item": "missing_value_representation", "value": "empty CSV field"},
    ]
)

parquet_export_contract_summary = pd.DataFrame(
    [
        {"contract_item": "processed_file", "value": str(CURATED_PARQUET_PATH.relative_to(PART_2_DIR))},
        {"contract_item": "source_table", "value": "curated_applications"},
        {"contract_item": "row_count", "value": f"{len(curated_applications):,}"},
        {"contract_item": "column_count", "value": str(len(LOCKED_CURATED_SCHEMA))},
        {"contract_item": "schema_order", "value": "LOCKED_CURATED_SCHEMA"},
        {"contract_item": "parquet_index", "value": "not exported"},
        {"contract_item": "engine", "value": PARQUET_EXPORT_ENGINE},
        {"contract_item": "compression", "value": PARQUET_EXPORT_COMPRESSION},
    ]
)

display(csv_export_contract_summary)
display(parquet_export_contract_summary)
print(
    "Validated curated CSV written to: "
    f"{CURATED_CSV_PATH.relative_to(PART_2_DIR)}"
)
print(
    "Validated curated Parquet written to: "
    f"{CURATED_PARQUET_PATH.relative_to(PART_2_DIR)}"
)


,contract_item,value
0,processed_file,data/processed/myh_curated_applications_2020_2...
1,source_table,curated_applications
2,row_count,"7,641"
3,column_count,32
4,schema_order,LOCKED_CURATED_SCHEMA
5,csv_index,not exported
6,encoding,utf-8
7,missing_value_representation,empty CSV field


,contract_item,value
0,processed_file,data/processed/myh_curated_applications_2020_2...
1,source_table,curated_applications
2,row_count,"7,641"
3,column_count,32
4,schema_order,LOCKED_CURATED_SCHEMA
5,parquet_index,not exported
6,engine,pyarrow
7,compression,snappy


Validated curated CSV written to: data/processed/myh_curated_applications_2020_2025.csv
Validated curated Parquet written to: data/processed/myh_curated_applications_2020_2025.parquet


### 11.2 Post-write CSV and Parquet verification

The in-memory quality gate proves that the curated DataFrame is export-ready. A final read-back check proves that the **saved files** are also usable: they exist, are non-empty, preserve the locked header order, retain the full year scope, and do not lose the application-grain key during serialization.

CSV is text-based, so its read-back check focuses on **file-level structural integrity** rather than trying to recover pandas extension dtypes exactly from a flat file. Parquet is the typed companion export, so its read-back check confirms that the second persisted artifact also preserves the same structural contract and key safety. The intended nullable dtype semantics remain documented and validated earlier in the curated-table checks.


In [25]:
csv_export_exists_ok = CURATED_CSV_PATH.exists()
csv_export_nonempty_ok = csv_export_exists_ok and CURATED_CSV_PATH.stat().st_size > 0
parquet_export_exists_ok = CURATED_PARQUET_PATH.exists()
parquet_export_nonempty_ok = parquet_export_exists_ok and CURATED_PARQUET_PATH.stat().st_size > 0

exported_csv_readback = pd.read_csv(CURATED_CSV_PATH, low_memory=False)
exported_parquet_readback = pd.read_parquet(
    CURATED_PARQUET_PATH,
    engine=PARQUET_EXPORT_ENGINE,
)

csv_readback_row_count_ok = len(exported_csv_readback) == len(curated_applications)
csv_readback_column_count_ok = exported_csv_readback.shape[1] == len(LOCKED_CURATED_SCHEMA)
csv_readback_schema_order_ok = list(exported_csv_readback.columns) == LOCKED_CURATED_SCHEMA
csv_readback_year_scope_ok = sorted(
    exported_csv_readback["source_year"].dropna().astype(int).unique().tolist()
) == EXPECTED_SOURCE_YEARS
csv_readback_no_index_artifact_ok = not any(
    str(column_name).startswith("Unnamed:") for column_name in exported_csv_readback.columns
)
csv_readback_diarienummer_complete_ok = int(exported_csv_readback["diarienummer"].isna().sum()) == 0
csv_readback_application_key_unique_ok = int(
    exported_csv_readback.duplicated(APPLICATION_GRAIN_KEY).sum()
) == 0

parquet_readback_row_count_ok = len(exported_parquet_readback) == len(curated_applications)
parquet_readback_column_count_ok = exported_parquet_readback.shape[1] == len(LOCKED_CURATED_SCHEMA)
parquet_readback_schema_order_ok = list(exported_parquet_readback.columns) == LOCKED_CURATED_SCHEMA
parquet_readback_year_scope_ok = sorted(
    exported_parquet_readback["source_year"].dropna().astype(int).unique().tolist()
) == EXPECTED_SOURCE_YEARS
parquet_readback_no_index_artifact_ok = not any(
    str(column_name).startswith("Unnamed:") or str(column_name).startswith("__index_level_")
    for column_name in exported_parquet_readback.columns
)
parquet_readback_diarienummer_complete_ok = int(exported_parquet_readback["diarienummer"].isna().sum()) == 0
parquet_readback_application_key_unique_ok = int(
    exported_parquet_readback.duplicated(APPLICATION_GRAIN_KEY).sum()
) == 0

csv_export_readback_validation_summary = pd.DataFrame(
    [
        {
            "check": "CSV export file exists",
            "passed": csv_export_exists_ok,
            "evidence": str(CURATED_CSV_PATH.relative_to(PART_2_DIR)),
        },
        {
            "check": "CSV export file is non-empty",
            "passed": csv_export_nonempty_ok,
            "evidence": f"{CURATED_CSV_PATH.stat().st_size:,} bytes" if csv_export_exists_ok else "missing file",
        },
        {
            "check": "CSV read-back preserves the curated row count",
            "passed": csv_readback_row_count_ok,
            "evidence": f"{len(exported_csv_readback):,} rows",
        },
        {
            "check": "CSV read-back preserves the curated column count",
            "passed": csv_readback_column_count_ok,
            "evidence": f"{exported_csv_readback.shape[1]} columns",
        },
        {
            "check": "CSV read-back preserves the locked header order",
            "passed": csv_readback_schema_order_ok,
            "evidence": "schema order matches LOCKED_CURATED_SCHEMA",
        },
        {
            "check": "CSV read-back preserves the 2020–2025 scope",
            "passed": csv_readback_year_scope_ok,
            "evidence": ", ".join(map(str, EXPECTED_SOURCE_YEARS)),
        },
        {
            "check": "CSV read-back contains no phantom exported index column",
            "passed": csv_readback_no_index_artifact_ok,
            "evidence": "no Unnamed: index artifact found",
        },
        {
            "check": "CSV read-back keeps diarienummer complete",
            "passed": csv_readback_diarienummer_complete_ok,
            "evidence": f"{int(exported_csv_readback['diarienummer'].isna().sum())} missing values",
        },
        {
            "check": "CSV read-back keeps the application-grain key unique",
            "passed": csv_readback_application_key_unique_ok,
            "evidence": f"{int(exported_csv_readback.duplicated(APPLICATION_GRAIN_KEY).sum())} duplicate key rows",
        },
    ]
)

parquet_export_readback_validation_summary = pd.DataFrame(
    [
        {
            "check": "Parquet export file exists",
            "passed": parquet_export_exists_ok,
            "evidence": str(CURATED_PARQUET_PATH.relative_to(PART_2_DIR)),
        },
        {
            "check": "Parquet export file is non-empty",
            "passed": parquet_export_nonempty_ok,
            "evidence": f"{CURATED_PARQUET_PATH.stat().st_size:,} bytes" if parquet_export_exists_ok else "missing file",
        },
        {
            "check": "Parquet read-back preserves the curated row count",
            "passed": parquet_readback_row_count_ok,
            "evidence": f"{len(exported_parquet_readback):,} rows",
        },
        {
            "check": "Parquet read-back preserves the curated column count",
            "passed": parquet_readback_column_count_ok,
            "evidence": f"{exported_parquet_readback.shape[1]} columns",
        },
        {
            "check": "Parquet read-back preserves the locked header order",
            "passed": parquet_readback_schema_order_ok,
            "evidence": "schema order matches LOCKED_CURATED_SCHEMA",
        },
        {
            "check": "Parquet read-back preserves the 2020–2025 scope",
            "passed": parquet_readback_year_scope_ok,
            "evidence": ", ".join(map(str, EXPECTED_SOURCE_YEARS)),
        },
        {
            "check": "Parquet read-back contains no phantom exported index column",
            "passed": parquet_readback_no_index_artifact_ok,
            "evidence": "no implicit index artifact found",
        },
        {
            "check": "Parquet read-back keeps diarienummer complete",
            "passed": parquet_readback_diarienummer_complete_ok,
            "evidence": f"{int(exported_parquet_readback['diarienummer'].isna().sum())} missing values",
        },
        {
            "check": "Parquet read-back keeps the application-grain key unique",
            "passed": parquet_readback_application_key_unique_ok,
            "evidence": f"{int(exported_parquet_readback.duplicated(APPLICATION_GRAIN_KEY).sum())} duplicate key rows",
        },
    ]
)

csv_export_roundtrip_ready_ok = bool(csv_export_readback_validation_summary["passed"].all())
parquet_export_roundtrip_ready_ok = bool(parquet_export_readback_validation_summary["passed"].all())
processed_exports_roundtrip_ready_ok = bool(
    csv_export_roundtrip_ready_ok and parquet_export_roundtrip_ready_ok
)

display(csv_export_readback_validation_summary)
display(parquet_export_readback_validation_summary)

assert csv_export_exists_ok, "The processed CSV file should exist after export."
assert csv_export_nonempty_ok, "The processed CSV file should be non-empty."
assert csv_readback_row_count_ok, "CSV read-back should preserve the curated row count."
assert csv_readback_column_count_ok, "CSV read-back should preserve the curated column count."
assert csv_readback_schema_order_ok, "CSV read-back should preserve the locked schema order."
assert csv_readback_year_scope_ok, "CSV read-back should preserve the 2020–2025 year scope."
assert csv_readback_no_index_artifact_ok, "CSV export should not contain a phantom index column."
assert csv_readback_diarienummer_complete_ok, "CSV read-back should keep diarienummer complete."
assert csv_readback_application_key_unique_ok, "CSV read-back should keep the application-grain key unique."
assert csv_export_roundtrip_ready_ok, "The CSV export must pass all post-write verification checks."

assert parquet_export_exists_ok, "The processed Parquet file should exist after export."
assert parquet_export_nonempty_ok, "The processed Parquet file should be non-empty."
assert parquet_readback_row_count_ok, "Parquet read-back should preserve the curated row count."
assert parquet_readback_column_count_ok, "Parquet read-back should preserve the curated column count."
assert parquet_readback_schema_order_ok, "Parquet read-back should preserve the locked schema order."
assert parquet_readback_year_scope_ok, "Parquet read-back should preserve the 2020–2025 year scope."
assert parquet_readback_no_index_artifact_ok, "Parquet export should not contain a phantom index column."
assert parquet_readback_diarienummer_complete_ok, "Parquet read-back should keep diarienummer complete."
assert parquet_readback_application_key_unique_ok, "Parquet read-back should keep the application-grain key unique."
assert parquet_export_roundtrip_ready_ok, "The Parquet export must pass all post-write verification checks."
assert processed_exports_roundtrip_ready_ok, "Both processed exports must pass the final post-write verification layer."


,check,passed,evidence
0,CSV export file exists,True,data/processed/myh_curated_applications_2020_2...
1,CSV export file is non-empty,True,"2,332,979 bytes"
2,CSV read-back preserves the curated row count,True,"7,641 rows"
3,CSV read-back preserves the curated column count,True,32 columns
4,CSV read-back preserves the locked header order,True,schema order matches LOCKED_CURATED_SCHEMA
5,CSV read-back preserves the 2020–2025 scope,True,"2020, 2021, 2022, 2023, 2024, 2025"
6,CSV read-back contains no phantom exported ind...,True,no Unnamed: index artifact found
7,CSV read-back keeps diarienummer complete,True,0 missing values
8,CSV read-back keeps the application-grain key ...,True,0 duplicate key rows


,check,passed,evidence
0,Parquet export file exists,True,data/processed/myh_curated_applications_2020_2...
1,Parquet export file is non-empty,True,"222,613 bytes"
2,Parquet read-back preserves the curated row count,True,"7,641 rows"
3,Parquet read-back preserves the curated column...,True,32 columns
4,Parquet read-back preserves the locked header ...,True,schema order matches LOCKED_CURATED_SCHEMA
5,Parquet read-back preserves the 2020–2025 scope,True,"2020, 2021, 2022, 2023, 2024, 2025"
6,Parquet read-back contains no phantom exported...,True,no implicit index artifact found
7,Parquet read-back keeps diarienummer complete,True,0 missing values
8,Parquet read-back keeps the application-grain ...,True,0 duplicate key rows


## 12. SQL/API handoff note and final reflection *(Sub-project 2.7)*

The notebook now ends with a practical handoff rather than treating export as the whole finish line. Part 2 produces a validated curated dataset; the later SQL and API work should consume that finished contract instead of reinterpreting the raw Excel files again.

### 12.1 What Part 3 can rely on

Later SQL loading and read-oriented API work can rely on the following Part 2 deliverables:

- **Processed inputs:**  
  - `part_2/data/processed/myh_curated_applications_2020_2025.csv`
  - `part_2/data/processed/myh_curated_applications_2020_2025.parquet`
- **Source table:** both exports come from the validated `curated_applications` DataFrame.
- **Grain:** one row = one application in one MYH application round.
- **Scope:** application rounds **2020–2025**.
- **Schema:** the locked **32-field** curated order used throughout the final notebook.
- **Traceability:** `source_year`, `source_file`, `source_sheet`, and `source_row` remain available for audit back to the raw workbooks.
- **Key safety:** the exported main-table grain stays unique through `(source_year, diarienummer)`.
- **Null policy:** structural nulls are retained where older source files did not contain later-year concepts; they are not disguised as zeroes or empty business values.

### 12.2 Practical SQL handoff

A later SQL step can treat the paired exports as alternate serialized forms of the same validated main table. The implementation can choose whichever format best fits the loader, but the database table should preserve the curated meanings established here.

The SQL phase can reasonably:
- load one main applications table from either export,
- preserve `(source_year, diarienummer)` as the natural uniqueness rule for the curated application grain,
- keep the source-traceability fields for audit and debugging,
- keep normalized fields such as `beslut_normalized` and `huvudmannatyp_normalized` for stable filtering/grouping,
- keep derived booleans such as `is_approved` and `is_distance_based` as convenient read-model fields,
- preserve nulls in structurally absent fields instead of replacing them during loading.

This notebook does **not** create the SQL schema itself. That belongs to the next project phase; Part 2’s job is to hand over a clear, validated dataset contract.

### 12.3 Practical read-oriented API handoff

A later FastAPI layer can build small read-oriented endpoints directly from the SQL table loaded from this curated dataset. The field choices in Part 2 already support common API questions such as:

- list applications for a selected `source_year`,
- retrieve one application by `(source_year, diarienummer)`,
- filter by `beslut_normalized`, `huvudmannatyp_normalized`, or `is_approved`,
- expose lightweight summaries by year or normalized decision category.

The API phase should expose curated meanings rather than raw workbook quirks. In other words, Part 2 absorbs source inconsistency so the API does not have to.

### 12.4 Final Part 2 reflection

Part 2 now demonstrates the full journey from messy multi-year source workbooks to a curated, quality-gated dataset suitable for later system use. The strongest design choices were:

- using `Tabell 3` as the single main-table backbone,
- refusing a casual `Tabell 4` merge that would violate the application grain,
- locking a 32-field curated schema before production ingestion,
- retaining raw meaning while adding normalized companion fields where cross-year comparison requires them,
- preserving structural nulls instead of fabricating values,
- validating the finished table before writing processed files,
- keeping both CSV and Parquet as complementary final exports.

The main limitations are also explicit rather than hidden:
- some fields exist only in later source years and therefore contain intentional structural nulls in earlier years,
- the reviewed `seqf_niva` gaps remain source-level missingness rather than notebook-created values,
- `Tabell 4` remains outside the main table because it has a different grain and would need a deliberate separate-table or aggregation design later.

These choices make the dataset more honest, more reproducible, and more useful for the SQL/API continuation than a denser but less defensible output would have been.

### 12.5 Part 2 definition-of-done check

Part 2 satisfies the project definition of done when the notebook:

- reads the six raw MYH Excel workbooks from `data/raw/`,
- explains the chosen years, the selected `Tabell 3` backbone, and the decision not to merge `Tabell 4`,
- documents the source structure, schema choices, harmonization rules, cleaning decisions, enrichment, and validation,
- builds the finished curated applications table with the intended grain and locked schema,
- exports the validated dataset as both CSV and Parquet,
- verifies the saved exports after writing,
- closes with a clear SQL/API handoff note and a concise reflection on the final curated result.

The preceding sections implement and demonstrate each of those points.


## Sub-project 2.7 completion checkpoint

The final Part 2 polish and handoff phase is complete when:
- the notebook reads as one coherent raw-to-curated data journey rather than a sequence of isolated implementation fragments,
- rerun expectations are clear near the start of the notebook and reflected in the validation/export flow,
- Section 12 states exactly what later SQL/API work can rely on from the curated dataset and paired exports,
- the final reflection explains why the dataset design is defensible, including the key limitations that remain intentionally visible,
- the Part 2 definition-of-done check is explicit and tied back to the implemented notebook,
- the notebook opening status block, repository-facing README, and end-of-session control files are advanced to the completed 2.7 state after the final full rerun/export confirmation.
